In [1]:
# ---------------- 1. Imports + paths ----------------

from pathlib import Path
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Main directories
RESULTS_DIR = Path("/home/jupyter/results/retrieval_subset")
CACHE_DIR = RESULTS_DIR / "cache"
INDEX_CACHE_DIR = CACHE_DIR / "index_frame_embeddings"
QUERY_CACHE_DIR = CACHE_DIR / "query_embeddings"

# Metadata / saved status files
N_CLIPS = 300
FINAL_CACHE_STATUS_PATH = RESULTS_DIR / f"cache_status_nclips{N_CLIPS}_final_after_repair.csv"
COMBINED_MANIFEST_PATH = RESULTS_DIR / f"preprocess_manifest_nclips{N_CLIPS}_all_batches.csv"

print("RESULTS_DIR:", RESULTS_DIR)
print("CACHE_DIR:", CACHE_DIR)
print("INDEX_CACHE_DIR:", INDEX_CACHE_DIR)
print("QUERY_CACHE_DIR:", QUERY_CACHE_DIR)

print("\nExists:")
print("RESULTS_DIR:", RESULTS_DIR.exists())
print("CACHE_DIR:", CACHE_DIR.exists())
print("INDEX_CACHE_DIR:", INDEX_CACHE_DIR.exists())
print("QUERY_CACHE_DIR:", QUERY_CACHE_DIR.exists())
print("FINAL_CACHE_STATUS_PATH:", FINAL_CACHE_STATUS_PATH.exists())
print("COMBINED_MANIFEST_PATH:", COMBINED_MANIFEST_PATH.exists())

RESULTS_DIR: /home/jupyter/results/retrieval_subset
CACHE_DIR: /home/jupyter/results/retrieval_subset/cache
INDEX_CACHE_DIR: /home/jupyter/results/retrieval_subset/cache/index_frame_embeddings
QUERY_CACHE_DIR: /home/jupyter/results/retrieval_subset/cache/query_embeddings

Exists:
RESULTS_DIR: False
CACHE_DIR: False
INDEX_CACHE_DIR: False
QUERY_CACHE_DIR: False
FINAL_CACHE_STATUS_PATH: False
COMBINED_MANIFEST_PATH: False


In [2]:
# ---------------- 2. Load 300 cached clip list ----------------

final_cache_status_df = pd.read_csv(FINAL_CACHE_STATUS_PATH)

print("Final cache status shape:", final_cache_status_df.shape)
display(final_cache_status_df.head())

# Use this file as the source of truth for the 300 clips.
all_300_clip_uids = final_cache_status_df["clip_uid"].astype(str).tolist()

print("Total clip rows:", len(all_300_clip_uids))
print("Unique clip_uids:", len(set(all_300_clip_uids)))

print("\nCache completeness:")
print("Index caches exist:", final_cache_status_df["index_cache_exists"].sum())
print("Query caches exist:", final_cache_status_df["query_cache_exists"].sum())
print("Query metadata exists:", final_cache_status_df["query_metadata_exists"].sum())
print("Fully cached:", final_cache_status_df["fully_cached"].sum())

missing_cache_df = final_cache_status_df[~final_cache_status_df["fully_cached"]].copy()
print("\nMissing / incomplete cache clips:", len(missing_cache_df))
display(missing_cache_df)

FileNotFoundError: [Errno 2] No such file or directory: '/home/jupyter/results/retrieval_subset/cache_status_nclips300_final_after_repair.csv'

In [ ]:
# ---------------- 3. Cache loading sanity check ----------------

def load_index_cache(clip_uid):
    path = INDEX_CACHE_DIR / f"{clip_uid}_index_embeddings.npz"
    data = np.load(path, allow_pickle=True)

    return {
        "path": path,
        "frame_numbers": data["frame_numbers"],
        "clip_embs": data["clip_embs"],
        "blip_embs": data["blip_embs"],
    }


def load_query_cache(clip_uid):
    emb_path = QUERY_CACHE_DIR / f"{clip_uid}_query_embeddings.npz"
    meta_path = QUERY_CACHE_DIR / f"{clip_uid}_query_metadata.parquet"

    data = np.load(emb_path, allow_pickle=True)
    query_metadata = pd.read_parquet(meta_path)

    return {
        "emb_path": emb_path,
        "meta_path": meta_path,
        "query_metadata": query_metadata,
        "clip_embs": data["clip_embs"],
        "blip_embs": data["blip_embs"],
    }


# Test one clip
test_clip_uid = all_300_clip_uids[0]

index_cache = load_index_cache(test_clip_uid)
query_cache = load_query_cache(test_clip_uid)

print("Test clip_uid:", test_clip_uid)

print("\nIndex cache:")
print("Frame numbers shape:", index_cache["frame_numbers"].shape)
print("CLIP index embedding shape:", index_cache["clip_embs"].shape)
print("BLIP index embedding shape:", index_cache["blip_embs"].shape)

print("\nQuery cache:")
print("Query metadata shape:", query_cache["query_metadata"].shape)
print("CLIP query embedding shape:", query_cache["clip_embs"].shape)
print("BLIP query embedding shape:", query_cache["blip_embs"].shape)

display(query_cache["query_metadata"].head())

Test clip_uid: dfe962ab-6aa7-4888-9378-796817a30ab6

Index cache:
Frame numbers shape: (300,)
CLIP index embedding shape: (300, 512)
BLIP index embedding shape: (300, 768)

Query cache:
Query metadata shape: (6, 5)
CLIP query embedding shape: (6, 512)
BLIP query embedding shape: (6, 768)


,clip_uid,annotation_uid,qs_id,object_title,vc_blob
0,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,2,portable fan,frames/retrieval/visual_crops/val/dfe962ab-6aa...
1,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,3,tube,frames/retrieval/visual_crops/val/dfe962ab-6aa...
2,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,1,green bowl,frames/retrieval/visual_crops/val/dfe962ab-6aa...
3,dfe962ab-6aa7-4888-9378-796817a30ab6,dc29a4cf-1430-43fe-98d2-156ca2b303cf,2,stool,frames/retrieval/visual_crops/val/dfe962ab-6aa...
4,dfe962ab-6aa7-4888-9378-796817a30ab6,dc29a4cf-1430-43fe-98d2-156ca2b303cf,3,bucket,frames/retrieval/visual_crops/val/dfe962ab-6aa...


In [ ]:
# ---------------- 4. Define success and retrieval helper functions ----------------

TOLERANCE_FRAMES = 5

def is_success(retrieved_frame, rt_min_frame, rt_max_frame, tolerance_frames=TOLERANCE_FRAMES):
    """
    Check whether retrieved frame falls inside the response-track range,
    allowing a small tolerance.
    """
    if pd.isna(retrieved_frame) or pd.isna(rt_min_frame) or pd.isna(rt_max_frame):
        return False

    return (
        retrieved_frame >= rt_min_frame - tolerance_frames
        and retrieved_frame <= rt_max_frame + tolerance_frames
    )


def compute_similarity_scores(index_embs, query_emb):
    """
    Embeddings are already L2-normalized.
    Dot product is cosine similarity.
    """
    return index_embs @ query_emb


def get_topk_frames(frame_numbers, scores, k=100):
    """
    Return top-k frame numbers and scores, sorted by descending score.
    """
    k = min(k, len(scores))
    top_idx = np.argsort(scores)[::-1][:k]

    return pd.DataFrame({
        "rank": np.arange(1, k + 1),
        "frame_number": frame_numbers[top_idx],
        "score": scores[top_idx],
    })


def first_correct_rank(topk_df, rt_min_frame, rt_max_frame, tolerance_frames=TOLERANCE_FRAMES):
    """
    Find the first rank where the retrieved frame is successful.
    Return np.nan if no correct frame appears in top-k.
    """
    for _, row in topk_df.iterrows():
        if is_success(
            retrieved_frame=row["frame_number"],
            rt_min_frame=rt_min_frame,
            rt_max_frame=rt_max_frame,
            tolerance_frames=tolerance_frames,
        ):
            return int(row["rank"])

    return np.nan


print("Helper functions defined.")
print("TOLERANCE_FRAMES:", TOLERANCE_FRAMES)

Helper functions defined.
TOLERANCE_FRAMES: 5


In [ ]:
# ---------------- 5. Check response-track lookup ----------------

print("rt_lookup exists:", "rt_lookup" in globals())

if "rt_lookup" in globals():
    print("Response-track entries:", len(rt_lookup))
    sample_key = next(iter(rt_lookup))
    print("Sample key:", sample_key)
    print("Sample value:", rt_lookup[sample_key])
else:
    print("rt_lookup is not defined yet.")
    print("We need to load or rebuild response-track labels before baseline retrieval.")

rt_lookup exists: False
rt_lookup is not defined yet.
We need to load or rebuild response-track labels before baseline retrieval.


In [ ]:
# ---------------- 6. Locate possible label / response-track files ----------------

from pathlib import Path

search_roots = [
    Path("/home/jupyter/results/retrieval_subset"),
    Path("/home/jupyter/egorecall_retrieval_subset"),
    Path("/home/jupyter/egorecall_retrieval_work"),
    Path("/home/jupyter/tmp/retrieval_work"),
    Path("/home/jupyter"),
]

keywords = [
    "response",
    "track",
    "query",
    "vq",
    "label",
    "metadata",
    "manifest",
]

matched_files = []

for root in search_roots:
    if not root.exists():
        continue

    for p in root.rglob("*"):
        if p.is_file():
            name_lower = p.name.lower()
            if any(k in name_lower for k in keywords):
                matched_files.append(p)

print("Matched files:", len(matched_files))

for p in matched_files[:200]:
    try:
        size_mb = p.stat().st_size / 1024 / 1024
        print(f"{size_mb:8.2f} MB | {p}")
    except Exception as e:
        print("Could not stat:", p, e)

Matched files: 1264
    0.01 MB | /home/jupyter/results/retrieval_subset/preprocess_manifest_nclips300_batch2.csv
    0.00 MB | /home/jupyter/results/retrieval_subset/preprocess_manifest_nclips300_batch0_sanity3.csv
    0.01 MB | /home/jupyter/results/retrieval_subset/preprocess_manifest_nclips300_batch4.csv
    0.01 MB | /home/jupyter/results/retrieval_subset/preprocess_manifest_nclips300_batch1.csv
    0.04 MB | /home/jupyter/results/retrieval_subset/preprocess_manifest_nclips300_all_batches.csv
    0.01 MB | /home/jupyter/results/retrieval_subset/preprocess_manifest_nclips300_batch3.csv
    0.01 MB | /home/jupyter/results/retrieval_subset/preprocess_manifest_nclips300_batch0.csv
    0.00 MB | /home/jupyter/results/retrieval_subset/cache/query_embeddings/09d85b55-d473-49c1-9ebc-e871714785cd_query_metadata.parquet
    0.01 MB | /home/jupyter/results/retrieval_subset/cache/query_embeddings/c2229daf-ec7b-45c5-84e3-bb4f4d145d93_query_embeddings.npz
    0.00 MB | /home/jupyter/results/ret

In [ ]:
# ---------------- 7. Inspect candidate table files ----------------

candidate_table_files = [
    p for p in matched_files
    if p.suffix.lower() in [".parquet", ".csv"]
]

print("Candidate table files:", len(candidate_table_files))

for p in candidate_table_files:
    print("=" * 100)
    print(p)

    try:
        if p.suffix.lower() == ".parquet":
            df_tmp = pd.read_parquet(p)
        else:
            df_tmp = pd.read_csv(p)

        print("Shape:", df_tmp.shape)
        print("Columns:", df_tmp.columns.tolist())
        display(df_tmp.head(3))

    except Exception as e:
        print("Failed to read:", repr(e))

Candidate table files: 662
/home/jupyter/results/retrieval_subset/preprocess_manifest_nclips300_batch2.csv
Shape: (60, 9)
Columns: ['clip_uid', 'video_uid', 'status', 'error', 'index_cache_exists_before', 'query_cache_exists_before', 'query_metadata_exists_before', 'num_index_frames', 'num_queries']


,clip_uid,video_uid,status,error,index_cache_exists_before,query_cache_exists_before,query_metadata_exists_before,num_index_frames,num_queries
0,617e8d7c-8764-4508-ba9b-c993282c70cd,25beabfc-48c0-4f81-b419-c47297aecf6f,skipped_fully_cached,NaN,True,True,True,NaN,3
1,b58bc49e-d3ac-4995-baa0-218a14e4c2fa,25beabfc-48c0-4f81-b419-c47297aecf6f,skipped_fully_cached,NaN,True,True,True,NaN,3
2,81221073-c095-4129-b095-97f3d6e4a6b1,412763ce-9a81-4bb7-b713-80fe2713e1a8,skipped_fully_cached,NaN,True,True,True,NaN,3


/home/jupyter/results/retrieval_subset/preprocess_manifest_nclips300_batch0_sanity3.csv
Shape: (3, 8)
Columns: ['clip_uid', 'video_uid', 'status', 'error', 'video_exists_before', 'index_cache_exists_before', 'query_cache_exists_before', 'query_metadata_exists_before']


,clip_uid,video_uid,status,error,video_exists_before,index_cache_exists_before,query_cache_exists_before,query_metadata_exists_before
0,159ac0ad-e5ec-449d-a69e-556655f79b2a,e14e03f8-13e4-4df2-87b0-e1ad8a175f7c,error,"AttributeError(""'BaseModelOutputWithPooling' o...",False,False,False,False
1,830e1b42-1d4a-4a4b-b0ad-c40fc4ace19c,eb486644-f46f-44cf-8e4b-73245b2fc02e,error,"AttributeError(""'BaseModelOutputWithPooling' o...",False,False,False,False
2,7bb41f34-1743-42ad-97d1-c54f4472b85e,eb486644-f46f-44cf-8e4b-73245b2fc02e,error,"AttributeError(""'BaseModelOutputWithPooling' o...",True,False,False,False


/home/jupyter/results/retrieval_subset/preprocess_manifest_nclips300_batch4.csv
Shape: (60, 9)
Columns: ['clip_uid', 'video_uid', 'status', 'error', 'index_cache_exists_before', 'query_cache_exists_before', 'query_metadata_exists_before', 'num_index_frames', 'num_queries']


...

,clip_uid,annotation_uid,qs_id,object_title,vc_blob,prompt
0,09d85b55-d473-49c1-9ebc-e871714785cd,29db216a-320b-4861-972a-d4a9ae719ced,1,calculator,frames/retrieval/visual_crops/val/09d85b55-d47...,a photo of a calculator
1,09d85b55-d473-49c1-9ebc-e871714785cd,29db216a-320b-4861-972a-d4a9ae719ced,2,tray,frames/retrieval/visual_crops/val/09d85b55-d47...,a photo of a tray
2,09d85b55-d473-49c1-9ebc-e871714785cd,29db216a-320b-4861-972a-d4a9ae719ced,3,spray bottle,frames/retrieval/visual_crops/val/09d85b55-d47...,a photo of a spray bottle


/home/jupyter/results/retrieval_subset/cache/clip_text_query_embeddings/e4728a65-3184-4239-9b84-be43dfd7377e_text_metadata_a_photo_of_a_object.parquet
Shape: (3, 6)
Columns: ['clip_uid', 'annotation_uid', 'qs_id', 'object_title', 'vc_blob', 'prompt']


,clip_uid,annotation_uid,qs_id,object_title,vc_blob,prompt
0,e4728a65-3184-4239-9b84-be43dfd7377e,310aef6e-4b04-4605-8192-0690451d94a3,2,jug,frames/retrieval/visual_crops/val/e4728a65-318...,a photo of a jug
1,e4728a65-3184-4239-9b84-be43dfd7377e,310aef6e-4b04-4605-8192-0690451d94a3,3,wood cutting machine,frames/retrieval/visual_crops/val/e4728a65-318...,a photo of a wood cutting machine
2,e4728a65-3184-4239-9b84-be43dfd7377e,310aef6e-4b04-4605-8192-0690451d94a3,1,container,frames/retrieval/visual_crops/val/e4728a65-318...,a photo of a container


/home/jupyter/results/retrieval_subset/cache/clip_text_query_embeddings/67c47998-eabe-4b31-9096-b286c18e1beb_text_metadata_a_photo_of_a_object.parquet
Shape: (6, 6)
Columns: ['clip_uid', 'annotation_uid', 'qs_id', 'object_title', 'vc_blob', 'prompt']


,clip_uid,annotation_uid,qs_id,object_title,vc_blob,prompt
0,67c47998-eabe-4b31-9096-b286c18e1beb,6cb2d772-cba3-4cf5-baba-5987319a75db,1,basket,frames/retrieval/visual_crops/val/67c47998-eab...,a photo of a basket
1,67c47998-eabe-4b31-9096-b286c18e1beb,6cb2d772-cba3-4cf5-baba-5987319a75db,2,wallet,frames/retrieval/visual_crops/val/67c47998-eab...,a photo of a wallet
2,67c47998-eabe-4b31-9096-b286c18e1beb,6cb2d772-cba3-4cf5-baba-5987319a75db,3,receipt,frames/retrieval/visual_crops/val/67c47998-eab...,a photo of a receipt


In [ ]:
# ---------------- 8. Focused search for source metadata files ----------------

from pathlib import Path
import pandas as pd

focus_roots = [
    Path("/home/jupyter"),
    Path("/home/jupyter/egorecall_retrieval_subset"),
    Path("/home/jupyter/results/retrieval_subset"),
]

target_names = [
    "vq_query_sets.parquet",
    "retrieval_metadata.parquet",
    "extraction_manifest.parquet",
    "response_track.parquet",
    "response_tracks.parquet",
]

focused_matches = []

for root in focus_roots:
    if not root.exists():
        continue

    for name in target_names:
        focused_matches.extend(list(root.rglob(name)))

# Remove duplicates while preserving order
seen = set()
focused_matches_unique = []
for p in focused_matches:
    if str(p) not in seen:
        focused_matches_unique.append(p)
        seen.add(str(p))

print("Focused matches:", len(focused_matches_unique))
for p in focused_matches_unique:
    print(p)

Focused matches: 4
/home/jupyter/egorecall_retrieval_work/vq_query_sets.parquet
/home/jupyter/egorecall_retrieval_subset/vq_query_sets.parquet
/home/jupyter/egorecall_retrieval_subset/retrieval_metadata.parquet
/home/jupyter/egorecall_retrieval_subset/extraction_manifest.parquet


In [ ]:
# ---------------- 9. Inspect focused metadata tables ----------------

for p in focused_matches_unique:
    print("=" * 120)
    print("File:", p)

    try:
        df_tmp = pd.read_parquet(p)
        print("Shape:", df_tmp.shape)
        print("Columns:", df_tmp.columns.tolist())
        display(df_tmp.head(5))

    except Exception as e:
        print("Failed to read:", repr(e))

File: /home/jupyter/egorecall_retrieval_work/vq_query_sets.parquet
Shape: (18114, 33)
Columns: ['split', 'video_uid', 'annotation_uid', 'qs_id', 'object_title', 'query_frame', 'query_video_frame', 'is_valid', 'has_errors', 'has_warnings', 'response_track_len', 'vc_frame', 'vc_x', 'vc_y', 'vc_w', 'vc_h', 'orig_w', 'orig_h', 'clip_uid', 'source_clip_uid', 'clip_fps', 'clip_start_sec', 'clip_end_sec', 'clip_duration_sec', 'video_start_sec', 'video_end_sec', 'annotation_complete', 'vc_area_norm', 'vc_cx_norm', 'vc_cy_norm', 'vc_aspect', 'clip_total_frames', 'query_pos_norm']


,split,video_uid,annotation_uid,qs_id,object_title,query_frame,query_video_frame,is_valid,has_errors,has_warnings,...,clip_duration_sec,video_start_sec,video_end_sec,annotation_complete,vc_area_norm,vc_cx_norm,vc_cy_norm,vc_aspect,clip_total_frames,query_pos_norm
0,train,56a3b49c-6979-4253-9ff5-2733c3e2f229,c64628fc-f163-4eb6-b7b1-f89b35cdcf54,1,mr krips,1190.0,15239.0,True,False,True,...,300.0,269.987695,569.987695,True,0.006133,0.542538,0.600069,0.643638,1500,0.793333
1,train,56a3b49c-6979-4253-9ff5-2733c3e2f229,c64628fc-f163-4eb6-b7b1-f89b35cdcf54,2,oats & more,1176.0,15155.0,True,False,True,...,300.0,269.987695,569.987695,True,0.052142,0.689042,0.591903,0.265028,1500,0.784000
2,train,56a3b49c-6979-4253-9ff5-2733c3e2f229,c64628fc-f163-4eb6-b7b1-f89b35cdcf54,3,tissue,1339.0,16133.0,True,False,True,...,300.0,269.987695,569.987695,True,0.009538,0.718073,0.338042,0.626356,1500,0.892667
3,train,56a3b49c-6979-4253-9ff5-2733c3e2f229,69b2bddb-8160-4779-9157-8696cb019bd2,1,basket,1090.0,14639.0,True,False,True,...,300.0,269.987695,569.987695,True,0.005293,0.069316,0.740477,0.910401,1500,0.726667
4,train,56a3b49c-6979-4253-9ff5-2733c3e2f229,69b2bddb-8160-4779-9157-8696cb019bd2,2,carton box,1283.0,15797.0,True,False,True,...,300.0,269.987695,569.987695,True,0.028745,0.814385,0.597898,1.443516,1500,0.855333


File: /home/jupyter/egorecall_retrieval_subset/vq_query_sets.parquet
Shape: (18114, 33)
Columns: ['split', 'video_uid', 'annotation_uid', 'qs_id', 'object_title', 'query_frame', 'query_video_frame', 'is_valid', 'has_errors', 'has_warnings', 'response_track_len', 'vc_frame', 'vc_x', 'vc_y', 'vc_w', 'vc_h', 'orig_w', 'orig_h', 'clip_uid', 'source_clip_uid', 'clip_fps', 'clip_start_sec', 'clip_end_sec', 'clip_duration_sec', 'video_start_sec', 'video_end_sec', 'annotation_complete', 'vc_area_norm', 'vc_cx_norm', 'vc_cy_norm', 'vc_aspect', 'clip_total_frames', 'query_pos_norm']


,split,video_uid,annotation_uid,qs_id,object_title,query_frame,query_video_frame,is_valid,has_errors,has_warnings,...,clip_duration_sec,video_start_sec,video_end_sec,annotation_complete,vc_area_norm,vc_cx_norm,vc_cy_norm,vc_aspect,clip_total_frames,query_pos_norm
0,train,56a3b49c-6979-4253-9ff5-2733c3e2f229,c64628fc-f163-4eb6-b7b1-f89b35cdcf54,1,mr krips,1190.0,15239.0,True,False,True,...,300.0,269.987695,569.987695,True,0.006133,0.542538,0.600069,0.643638,1500,0.793333
1,train,56a3b49c-6979-4253-9ff5-2733c3e2f229,c64628fc-f163-4eb6-b7b1-f89b35cdcf54,2,oats & more,1176.0,15155.0,True,False,True,...,300.0,269.987695,569.987695,True,0.052142,0.689042,0.591903,0.265028,1500,0.784000
2,train,56a3b49c-6979-4253-9ff5-2733c3e2f229,c64628fc-f163-4eb6-b7b1-f89b35cdcf54,3,tissue,1339.0,16133.0,True,False,True,...,300.0,269.987695,569.987695,True,0.009538,0.718073,0.338042,0.626356,1500,0.892667
3,train,56a3b49c-6979-4253-9ff5-2733c3e2f229,69b2bddb-8160-4779-9157-8696cb019bd2,1,basket,1090.0,14639.0,True,False,True,...,300.0,269.987695,569.987695,True,0.005293,0.069316,0.740477,0.910401,1500,0.726667
4,train,56a3b49c-6979-4253-9ff5-2733c3e2f229,69b2bddb-8160-4779-9157-8696cb019bd2,2,carton box,1283.0,15797.0,True,False,True,...,300.0,269.987695,569.987695,True,0.028745,0.814385,0.597898,1.443516,1500,0.855333


File: /home/jupyter/egorecall_retrieval_subset/retrieval_metadata.parquet
Failed to read: ArrowInvalid("Could not open Parquet input source '<Buffer>': Parquet file size is 0 bytes")
File: /home/jupyter/egorecall_retrieval_subset/extraction_manifest.parquet
Shape: (1743, 7)
Columns: ['video_uid', 'split', 'n_clips', 'detection_frames', 'yolo_labels', 'visual_crops', 'status']


,video_uid,split,n_clips,detection_frames,yolo_labels,visual_crops,status
0,56a3b49c-6979-4253-9ff5-2733c3e2f229,train,5.0,281.0,281.0,24.0,ok
1,a7a6e05d-ffe6-47a4-a08c-b4b386e9911f,train,2.0,55.0,55.0,6.0,ok
2,ee114c86-05fc-4d47-8c9a-8dfeafed0457,train,2.0,140.0,140.0,9.0,ok
3,95710114-4168-47b4-a63e-a4d220b42fcf,train,1.0,36.0,36.0,3.0,ok
4,6146205c-f045-4368-acc2-a60dc5e1490b,train,1.0,154.0,154.0,3.0,ok


In [ ]:
# ---------------- 10. Inspect vq_query_sets response-track fields ----------------

VQ_QUERY_PATH = Path("/home/jupyter/egorecall_retrieval_subset/vq_query_sets.parquet")

vq_df = pd.read_parquet(VQ_QUERY_PATH)

print("vq_df shape:", vq_df.shape)
print("\nAll columns:")
for i, col in enumerate(vq_df.columns):
    print(i, col)

print("\nColumns containing frame / track / response / vc / query:")
key_cols = [
    c for c in vq_df.columns
    if any(k in c.lower() for k in ["frame", "track", "response", "vc", "query", "clip"])
]
for c in key_cols:
    print(c)

display(vq_df[key_cols].head(10))

vq_df shape: (18114, 33)

All columns:
0 split
1 video_uid
2 annotation_uid
3 qs_id
4 object_title
5 query_frame
6 query_video_frame
7 is_valid
8 has_errors
9 has_warnings
10 response_track_len
11 vc_frame
12 vc_x
13 vc_y
14 vc_w
15 vc_h
16 orig_w
17 orig_h
18 clip_uid
19 source_clip_uid
20 clip_fps
21 clip_start_sec
22 clip_end_sec
23 clip_duration_sec
24 video_start_sec
25 video_end_sec
26 annotation_complete
27 vc_area_norm
28 vc_cx_norm
29 vc_cy_norm
30 vc_aspect
31 clip_total_frames
32 query_pos_norm

Columns containing frame / track / response / vc / query:
query_frame
query_video_frame
response_track_len
vc_frame
vc_x
vc_y
vc_w
vc_h
clip_uid
source_clip_uid
clip_fps
clip_start_sec
clip_end_sec
clip_duration_sec
vc_area_norm
vc_cx_norm
vc_cy_norm
vc_aspect
clip_total_frames
query_pos_norm


,query_frame,query_video_frame,response_track_len,vc_frame,vc_x,vc_y,vc_w,vc_h,clip_uid,source_clip_uid,clip_fps,clip_start_sec,clip_end_sec,clip_duration_sec,vc_area_norm,vc_cx_norm,vc_cy_norm,vc_aspect,clip_total_frames,query_pos_norm
0,1190.0,15239.0,11,1198.0,742.08,587.21,78.35,121.73,3fe39989-54ed-4a3a-bea4-88be05ff8df7,e970b901-fd09-4e00-ad86-12bbc04b02fb,5.0,0.0,300.000000,300.000000,0.006133,0.542538,0.600069,0.643638,1500,0.793333
1,1176.0,15155.0,13,1182.0,918.92,362.68,146.60,553.15,3fe39989-54ed-4a3a-bea4-88be05ff8df7,e970b901-fd09-4e00-ad86-12bbc04b02fb,5.0,0.0,300.000000,300.000000,0.052142,0.689042,0.591903,0.265028,1500,0.784000
2,1339.0,16133.0,9,1356.0,985.83,288.14,96.39,153.89,3fe39989-54ed-4a3a-bea4-88be05ff8df7,e970b901-fd09-4e00-ad86-12bbc04b02fb,5.0,0.0,300.000000,300.000000,0.009538,0.718073,0.338042,0.626356,1500,0.892667
3,1090.0,14639.0,15,1053.0,56.53,752.17,86.57,95.09,3fe39989-54ed-4a3a-bea4-88be05ff8df7,e970b901-fd09-4e00-ad86-12bbc04b02fb,5.0,0.0,300.000000,300.000000,0.005293,0.069316,0.740477,0.910401,1500,0.726667
4,1283.0,15797.0,12,1111.0,1045.70,557.74,254.03,175.98,3fe39989-54ed-4a3a-bea4-88be05ff8df7,e970b901-fd09-4e00-ad86-12bbc04b02fb,5.0,0.0,300.000000,300.000000,0.028745,0.814385,0.597898,1.443516,1500,0.855333
5,1315.0,15989.0,13,1284.0,1187.62,543.55,171.72,228.49,3fe39989-54ed-4a3a-bea4-88be05ff8df7,e970b901-fd09-4e00-ad86-12bbc04b02fb,5.0,0.0,300.000000,300.000000,0.025229,0.884361,0.609069,0.751543,1500,0.876667
6,113.0,677.0,8,114.0,1308.87,909.47,73.28,114.99,769b357e-508a-4741-a93f-642bb14dd2a0,011ebb4e-e7fa-47a9-bf36-453cfa015053,5.0,0.0,959.966667,959.966667,0.005418,0.934382,0.895338,0.637273,4800,0.023542
7,513.0,3077.0,11,49.0,146.89,722.63,95.20,186.47,769b357e-508a-4741-a93f-642bb14dd2a0,011ebb4e-e7fa-47a9-bf36-453cfa015053,5.0,0.0,959.966667,959.966667,0.011415,0.135062,0.755431,0.510538,4800,0.106875
8,384.0,2303.0,9,394.0,726.71,582.80,227.59,206.77,769b357e-508a-4741-a93f-642bb14dd2a0,011ebb4e-e7fa-47a9-bf36-453cfa015053,5.0,0.0,959.966667,959.966667,0.030259,0.583684,0.635356,1.100692,4800,0.080000
9,325.0,18149.0,11,348.0,609.61,646.73,117.90,168.00,bdc60480-9ef7-4092-b470-4a75368a312c,84cba418-7727-4504-92af-30112ce27a7b,5.0,0.0,300.000000,300.000000,0.012736,0.464278,0.676602,0.701786,1500,0.216667


In [ ]:
# ---------------- 11. Filter vq_query_sets to our 300 cached clips ----------------

vq_300_df = vq_df[vq_df["clip_uid"].astype(str).isin(all_300_clip_uids)].copy()

print("vq_300_df shape:", vq_300_df.shape)
print("Unique clips in vq_300_df:", vq_300_df["clip_uid"].nunique())

display(
    vq_300_df[
        [
            "clip_uid",
            "annotation_uid",
            "qs_id",
            "object_title",
            "query_frame",
            "query_video_frame",
            "response_track_len",
            "clip_total_frames",
        ]
    ].head(20)
)

vq_300_df shape: (1157, 33)
Unique clips in vq_300_df: 300


,clip_uid,annotation_uid,qs_id,object_title,query_frame,query_video_frame,response_track_len,clip_total_frames
13610,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,2,portable fan,298.0,42288.0,30,1500
13611,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,3,tube,1044.0,46764.0,7,1500
13612,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,1,green bowl,1498.0,49488.0,49,1500
13613,dfe962ab-6aa7-4888-9378-796817a30ab6,dc29a4cf-1430-43fe-98d2-156ca2b303cf,2,stool,1499.0,49494.0,18,1500
13614,dfe962ab-6aa7-4888-9378-796817a30ab6,dc29a4cf-1430-43fe-98d2-156ca2b303cf,3,bucket,363.0,42678.0,13,1500
13615,dfe962ab-6aa7-4888-9378-796817a30ab6,dc29a4cf-1430-43fe-98d2-156ca2b303cf,1,stool,489.0,43434.0,8,1500
13622,b157270d-c67a-41db-91ec-32bb3ec738fc,dd64c5e8-7679-4068-acc6-50b21970e2e0,1,dispensing bottle,827.0,29262.0,3,1500
13623,b157270d-c67a-41db-91ec-32bb3ec738fc,dd64c5e8-7679-4068-acc6-50b21970e2e0,3,adapter,674.0,28344.0,8,1500
13624,b157270d-c67a-41db-91ec-32bb3ec738fc,dd64c5e8-7679-4068-acc6-50b21970e2e0,2,impact driver,963.0,30078.0,9,1500
13625,b157270d-c67a-41db-91ec-32bb3ec738fc,3cedcbbe-7dd0-4ef0-b0ac-e7f26a4c20cb,2,orange towel,125.0,25050.0,33,1500


In [ ]:
# ---------------- 12. Build response-track labels / rt_lookup ----------------

VQ_QUERY_PATH = Path("/home/jupyter/egorecall_retrieval_subset/vq_query_sets.parquet")

vq_df = pd.read_parquet(VQ_QUERY_PATH)

# Keep only rows for our 300 cached clips
vq_300_df = vq_df[vq_df["clip_uid"].astype(str).isin(all_300_clip_uids)].copy()

# Make key columns consistent
vq_300_df["clip_uid"] = vq_300_df["clip_uid"].astype(str)
vq_300_df["annotation_uid"] = vq_300_df["annotation_uid"].astype(str)
vq_300_df["qs_id"] = vq_300_df["qs_id"].astype(int)

# Build response-track range in video-frame coordinates
vq_300_df["rt_min_frame"] = vq_300_df["query_video_frame"].astype(float)
vq_300_df["rt_max_frame"] = (
    vq_300_df["query_video_frame"].astype(float)
    + vq_300_df["response_track_len"].astype(float)
    - 1
)

# Keep only valid rows
label_df = vq_300_df[
    (vq_300_df["is_valid"] == True)
    & (vq_300_df["has_errors"] == False)
    & vq_300_df["query_video_frame"].notna()
    & vq_300_df["response_track_len"].notna()
].copy()

label_cols = [
    "clip_uid",
    "annotation_uid",
    "qs_id",
    "object_title",
    "query_frame",
    "query_video_frame",
    "response_track_len",
    "rt_min_frame",
    "rt_max_frame",
    "clip_total_frames",
]

label_df = label_df[label_cols].copy()

print("label_df shape:", label_df.shape)
print("Unique clips in label_df:", label_df["clip_uid"].nunique())
print("Unique query labels:", label_df[["clip_uid", "annotation_uid", "qs_id"]].drop_duplicates().shape[0])

display(label_df.head(10))

# Build lookup: key = (clip_uid, annotation_uid, qs_id)
rt_lookup = {}

for _, row in label_df.iterrows():
    key = (
        str(row["clip_uid"]),
        str(row["annotation_uid"]),
        int(row["qs_id"]),
    )
    rt_lookup[key] = {
        "object_title": row["object_title"],
        "query_frame": float(row["query_frame"]),
        "query_video_frame": float(row["query_video_frame"]),
        "response_track_len": float(row["response_track_len"]),
        "rt_min_frame": float(row["rt_min_frame"]),
        "rt_max_frame": float(row["rt_max_frame"]),
    }

print("\nrt_lookup entries:", len(rt_lookup))

sample_key = next(iter(rt_lookup))
print("Sample key:", sample_key)
print("Sample value:", rt_lookup[sample_key])

label_df shape: (1157, 10)
Unique clips in label_df: 300
Unique query labels: 1157


,clip_uid,annotation_uid,qs_id,object_title,query_frame,query_video_frame,response_track_len,rt_min_frame,rt_max_frame,clip_total_frames
13610,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,2,portable fan,298.0,42288.0,30,42288.0,42317.0,1500
13611,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,3,tube,1044.0,46764.0,7,46764.0,46770.0,1500
13612,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,1,green bowl,1498.0,49488.0,49,49488.0,49536.0,1500
13613,dfe962ab-6aa7-4888-9378-796817a30ab6,dc29a4cf-1430-43fe-98d2-156ca2b303cf,2,stool,1499.0,49494.0,18,49494.0,49511.0,1500
13614,dfe962ab-6aa7-4888-9378-796817a30ab6,dc29a4cf-1430-43fe-98d2-156ca2b303cf,3,bucket,363.0,42678.0,13,42678.0,42690.0,1500
13615,dfe962ab-6aa7-4888-9378-796817a30ab6,dc29a4cf-1430-43fe-98d2-156ca2b303cf,1,stool,489.0,43434.0,8,43434.0,43441.0,1500
13622,b157270d-c67a-41db-91ec-32bb3ec738fc,dd64c5e8-7679-4068-acc6-50b21970e2e0,1,dispensing bottle,827.0,29262.0,3,29262.0,29264.0,1500
13623,b157270d-c67a-41db-91ec-32bb3ec738fc,dd64c5e8-7679-4068-acc6-50b21970e2e0,3,adapter,674.0,28344.0,8,28344.0,28351.0,1500
13624,b157270d-c67a-41db-91ec-32bb3ec738fc,dd64c5e8-7679-4068-acc6-50b21970e2e0,2,impact driver,963.0,30078.0,9,30078.0,30086.0,1500
13625,b157270d-c67a-41db-91ec-32bb3ec738fc,3cedcbbe-7dd0-4ef0-b0ac-e7f26a4c20cb,2,orange towel,125.0,25050.0,33,25050.0,25082.0,1500



rt_lookup entries: 1157
Sample key: ('dfe962ab-6aa7-4888-9378-796817a30ab6', '3b057788-ad90-432d-90ff-de1c45361df6', 2)
Sample value: {'object_title': 'portable fan', 'query_frame': 298.0, 'query_video_frame': 42288.0, 'response_track_len': 30.0, 'rt_min_frame': 42288.0, 'rt_max_frame': 42317.0}


In [ ]:
# ---------------- 13. Verify query cache metadata can match rt_lookup ----------------

all_query_meta_rows = []

for clip_uid in tqdm(all_300_clip_uids, desc="Loading query metadata"):
    query_cache = load_query_cache(clip_uid)
    qmeta = query_cache["query_metadata"].copy()
    qmeta["clip_uid"] = qmeta["clip_uid"].astype(str)
    qmeta["annotation_uid"] = qmeta["annotation_uid"].astype(str)
    qmeta["qs_id"] = qmeta["qs_id"].astype(int)
    all_query_meta_rows.append(qmeta)

all_query_meta_df = pd.concat(all_query_meta_rows, ignore_index=True)

print("All query metadata shape:", all_query_meta_df.shape)
print("Unique clips:", all_query_meta_df["clip_uid"].nunique())
print("Unique query rows:", all_query_meta_df[["clip_uid", "annotation_uid", "qs_id"]].drop_duplicates().shape[0])

# Add lookup match flag
def has_rt_label(row):
    key = (
        str(row["clip_uid"]),
        str(row["annotation_uid"]),
        int(row["qs_id"]),
    )
    return key in rt_lookup

all_query_meta_df["has_rt_label"] = all_query_meta_df.apply(has_rt_label, axis=1)

print("\nRT label match count:")
print(all_query_meta_df["has_rt_label"].value_counts(dropna=False))

missing_rt_df = all_query_meta_df[~all_query_meta_df["has_rt_label"]].copy()

print("\nMissing RT labels:", len(missing_rt_df))
display(missing_rt_df.head(20))

Loading query metadata:   0%|          | 0/300 [00:00<?, ?it/s]

All query metadata shape: (1157, 5)
Unique clips: 300
Unique query rows: 1157

RT label match count:
has_rt_label
True    1157
Name: count, dtype: int64

Missing RT labels: 0


,clip_uid,annotation_uid,qs_id,object_title,vc_blob,has_rt_label


In [ ]:
# ---------------- 14. Baseline retrieval sanity check on one clip ----------------

def evaluate_one_clip_baseline(clip_uid, model_name="clip", topk=100, tolerance_frames=TOLERANCE_FRAMES):
    """
    Evaluate retrieval for one clip using cached embeddings.

    model_name:
        - "clip": use CLIP embeddings
        - "blip": use BLIP embeddings
    """
    index_cache = load_index_cache(clip_uid)
    query_cache = load_query_cache(clip_uid)

    frame_numbers = index_cache["frame_numbers"]

    if model_name == "clip":
        index_embs = index_cache["clip_embs"]
        query_embs = query_cache["clip_embs"]
    elif model_name == "blip":
        index_embs = index_cache["blip_embs"]
        query_embs = query_cache["blip_embs"]
    else:
        raise ValueError("model_name must be 'clip' or 'blip'")

    qmeta = query_cache["query_metadata"].copy()
    qmeta["clip_uid"] = qmeta["clip_uid"].astype(str)
    qmeta["annotation_uid"] = qmeta["annotation_uid"].astype(str)
    qmeta["qs_id"] = qmeta["qs_id"].astype(int)

    rows = []

    for q_idx, qrow in qmeta.iterrows():
        key = (
            str(qrow["clip_uid"]),
            str(qrow["annotation_uid"]),
            int(qrow["qs_id"]),
        )

        if key not in rt_lookup:
            continue

        rt = rt_lookup[key]
        query_emb = query_embs[q_idx]

        scores = compute_similarity_scores(index_embs, query_emb)
        topk_df = get_topk_frames(frame_numbers, scores, k=topk)

        rank = first_correct_rank(
            topk_df=topk_df,
            rt_min_frame=rt["rt_min_frame"],
            rt_max_frame=rt["rt_max_frame"],
            tolerance_frames=tolerance_frames,
        )

        top1_frame = int(topk_df.iloc[0]["frame_number"])
        top1_score = float(topk_df.iloc[0]["score"])

        rows.append({
            "clip_uid": clip_uid,
            "annotation_uid": qrow["annotation_uid"],
            "qs_id": int(qrow["qs_id"]),
            "object_title": qrow["object_title"],
            "model_name": model_name,
            "top1_frame": top1_frame,
            "top1_score": top1_score,
            "rt_min_frame": rt["rt_min_frame"],
            "rt_max_frame": rt["rt_max_frame"],
            "first_correct_rank": rank,
            "top1_success": bool(rank == 1),
            "top5_success": bool(pd.notna(rank) and rank <= 5),
            "top10_success": bool(pd.notna(rank) and rank <= 10),
            "top20_success": bool(pd.notna(rank) and rank <= 20),
            "top50_success": bool(pd.notna(rank) and rank <= 50),
            "top100_success": bool(pd.notna(rank) and rank <= 100),
        })

    return pd.DataFrame(rows)


test_clip_uid = all_300_clip_uids[0]

clip_test_df = evaluate_one_clip_baseline(
    clip_uid=test_clip_uid,
    model_name="clip",
    topk=100,
)

print("Test clip:", test_clip_uid)
print("Rows:", len(clip_test_df))
display(clip_test_df)

Test clip: dfe962ab-6aa7-4888-9378-796817a30ab6
Rows: 6


,clip_uid,annotation_uid,qs_id,object_title,model_name,top1_frame,top1_score,rt_min_frame,rt_max_frame,first_correct_rank,top1_success,top5_success,top10_success,top20_success,top50_success,top100_success
0,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,2,portable fan,clip,43470,0.820488,42288.0,42317.0,88.0,False,False,False,False,False,True
1,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,3,tube,clip,41370,0.795074,46764.0,46770.0,5.0,False,True,True,True,True,True
2,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,1,green bowl,clip,41370,0.795995,49488.0,49536.0,NaN,False,False,False,False,False,False
3,dfe962ab-6aa7-4888-9378-796817a30ab6,dc29a4cf-1430-43fe-98d2-156ca2b303cf,2,stool,clip,46770,0.852550,49494.0,49511.0,NaN,False,False,False,False,False,False
4,dfe962ab-6aa7-4888-9378-796817a30ab6,dc29a4cf-1430-43fe-98d2-156ca2b303cf,3,bucket,clip,46770,0.883756,42678.0,42690.0,53.0,False,False,False,False,False,True
5,dfe962ab-6aa7-4888-9378-796817a30ab6,dc29a4cf-1430-43fe-98d2-156ca2b303cf,1,stool,clip,44400,0.855764,43434.0,43441.0,79.0,False,False,False,False,False,True


In [ ]:
# ---------------- 15. BLIP baseline sanity check on one clip ----------------

blip_test_df = evaluate_one_clip_baseline(
    clip_uid=test_clip_uid,
    model_name="blip",
    topk=100,
)

print("Test clip:", test_clip_uid)
print("Rows:", len(blip_test_df))
display(blip_test_df)

Test clip: dfe962ab-6aa7-4888-9378-796817a30ab6
Rows: 6


,clip_uid,annotation_uid,qs_id,object_title,model_name,top1_frame,top1_score,rt_min_frame,rt_max_frame,first_correct_rank,top1_success,top5_success,top10_success,top20_success,top50_success,top100_success
0,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,2,portable fan,blip,42390,0.753312,42288.0,42317.0,73.0,False,False,False,False,False,True
1,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,3,tube,blip,41370,0.622319,46764.0,46770.0,3.0,False,True,True,True,True,True
2,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,1,green bowl,blip,47670,0.650754,49488.0,49536.0,NaN,False,False,False,False,False,False
3,dfe962ab-6aa7-4888-9378-796817a30ab6,dc29a4cf-1430-43fe-98d2-156ca2b303cf,2,stool,blip,46770,0.703079,49494.0,49511.0,NaN,False,False,False,False,False,False
4,dfe962ab-6aa7-4888-9378-796817a30ab6,dc29a4cf-1430-43fe-98d2-156ca2b303cf,3,bucket,blip,42300,0.721368,42678.0,42690.0,47.0,False,False,False,False,True,True
5,dfe962ab-6aa7-4888-9378-796817a30ab6,dc29a4cf-1430-43fe-98d2-156ca2b303cf,1,stool,blip,42480,0.714370,43434.0,43441.0,NaN,False,False,False,False,False,False


In [ ]:
# ---------------- 16. Run full baseline: CLIP + BLIP over 300 clips ----------------

baseline_rows = []

for model_name in ["clip", "blip"]:
    print("=" * 80)
    print("Running baseline:", model_name)

    for clip_uid in tqdm(all_300_clip_uids, desc=f"{model_name} baseline"):
        df_one = evaluate_one_clip_baseline(
            clip_uid=clip_uid,
            model_name=model_name,
            topk=100,
        )
        baseline_rows.append(df_one)

baseline_df = pd.concat(baseline_rows, ignore_index=True)

print("Baseline rows:", len(baseline_df))
print("Models:", baseline_df["model_name"].value_counts())

display(baseline_df.head())

baseline_path = RESULTS_DIR / f"baseline_clip_blip_nclips{N_CLIPS}_top100.csv"
baseline_df.to_csv(baseline_path, index=False)

print("Saved baseline results to:", baseline_path)

Running baseline: clip


clip baseline:   0%|          | 0/300 [00:00<?, ?it/s]

Running baseline: blip


blip baseline:   0%|          | 0/300 [00:00<?, ?it/s]

Baseline rows: 2314
Models: model_name
clip    1157
blip    1157
Name: count, dtype: int64


,clip_uid,annotation_uid,qs_id,object_title,model_name,top1_frame,top1_score,rt_min_frame,rt_max_frame,first_correct_rank,top1_success,top5_success,top10_success,top20_success,top50_success,top100_success
0,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,2,portable fan,clip,43470,0.820488,42288.0,42317.0,88.0,False,False,False,False,False,True
1,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,3,tube,clip,41370,0.795074,46764.0,46770.0,5.0,False,True,True,True,True,True
2,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,1,green bowl,clip,41370,0.795995,49488.0,49536.0,NaN,False,False,False,False,False,False
3,dfe962ab-6aa7-4888-9378-796817a30ab6,dc29a4cf-1430-43fe-98d2-156ca2b303cf,2,stool,clip,46770,0.852550,49494.0,49511.0,NaN,False,False,False,False,False,False
4,dfe962ab-6aa7-4888-9378-796817a30ab6,dc29a4cf-1430-43fe-98d2-156ca2b303cf,3,bucket,clip,46770,0.883756,42678.0,42690.0,53.0,False,False,False,False,False,True


Saved baseline results to: /home/jupyter/results/retrieval_subset/baseline_clip_blip_nclips300_top100.csv


In [ ]:
# ---------------- 17. Summarize baseline metrics ----------------

metric_cols = [
    "top1_success",
    "top5_success",
    "top10_success",
    "top20_success",
    "top50_success",
    "top100_success",
]

baseline_summary = (
    baseline_df
    .groupby("model_name")[metric_cols]
    .mean()
    .reset_index()
)

# Convert to percentage
for col in metric_cols:
    baseline_summary[col] = baseline_summary[col] * 100

display(baseline_summary)

summary_path = RESULTS_DIR / f"baseline_summary_nclips{N_CLIPS}_top100.csv"
baseline_summary.to_csv(summary_path, index=False)

print("Saved baseline summary to:", summary_path)

,model_name,top1_success,top5_success,top10_success,top20_success,top50_success,top100_success
0,blip,0.518583,1.469317,2.852204,5.617978,13.569576,25.669836
1,clip,0.518583,1.728608,2.938634,5.272256,15.038894,24.978392


Saved baseline summary to: /home/jupyter/results/retrieval_subset/baseline_summary_nclips300_top100.csv


In [ ]:
# ---------------- 18. First correct rank distribution ----------------

for model_name in ["clip", "blip"]:
    df_m = baseline_df[baseline_df["model_name"] == model_name].copy()

    print("=" * 80)
    print("Model:", model_name)
    print("Total queries:", len(df_m))
    print("Queries with correct frame in top100:", df_m["first_correct_rank"].notna().sum())
    print("Queries without correct frame in top100:", df_m["first_correct_rank"].isna().sum())

    display(
        df_m["first_correct_rank"]
        .describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95])
    )

Model: clip
Total queries: 1157
Queries with correct frame in top100: 289
Queries without correct frame in top100: 868


count    289.000000
mean      45.719723
std       27.021487
min        1.000000
25%       26.000000
50%       45.000000
75%       66.000000
90%       86.000000
95%       91.200000
max      100.000000
Name: first_correct_rank, dtype: float64

Model: blip
Total queries: 1157
Queries with correct frame in top100: 297
Queries without correct frame in top100: 860


count    297.000000
mean      49.070707
std       29.141634
min        1.000000
25%       23.000000
50%       48.000000
75%       73.000000
90%       91.000000
95%       95.000000
max      100.000000
Name: first_correct_rank, dtype: float64

## Baseline Retrieval Results: CLIP-only vs BLIP-only

We evaluated cached retrieval embeddings on 300 clips, covering 1,157 query/object cases.  
For each query, the system ranks cached index frames by cosine similarity and checks whether the retrieved frame falls within the ground-truth response-track frame range.

### Baseline Summary

| Model | Top-1 | Top-5 | Top-10 | Top-20 | Top-50 | Top-100 |
|---|---:|---:|---:|---:|---:|---:|
| BLIP | 0.52% | 1.47% | 2.85% | 5.62% | 13.57% | 25.67% |
| CLIP | 0.52% | 1.73% | 2.94% | 5.27% | 15.04% | 24.98% |

### Interpretation

The baseline retrieval performance is low, especially at Top-1. Both CLIP-only and BLIP-only retrieval achieve only around 0.52% Top-1 accuracy. Even at Top-100, the correct response-track frame appears for only about 25% of the queries.

This suggests that the main bottleneck is not only final ranking quality, but also candidate recall: for many queries, the correct frame does not appear in the top candidate set. Therefore, simple reranking may have limited impact unless the correct frame is already included among the retrieved candidates.

CLIP and BLIP perform similarly overall. CLIP is slightly better at Top-50, while BLIP is slightly better at Top-100. This suggests that the two embedding spaces may contain partially complementary signals, making CLIP/BLIP score fusion a reasonable next experiment.

Before applying more complex reranking methods, we should verify whether the sampled index frames actually cover the ground-truth response-track windows. If many response tracks are not represented by any sampled index frame, retrieval cannot succeed regardless of the ranking model.

In [ ]:
# ---------------- 19. Ground-truth coverage upper bound check ----------------

def check_gt_frame_coverage_for_one_clip(
    clip_uid,
    tolerance_frames=TOLERANCE_FRAMES,
):
    """
    Check whether each query's ground-truth response-track range
    is covered by at least one sampled index frame.
    """
    index_cache = load_index_cache(clip_uid)
    frame_numbers = index_cache["frame_numbers"]

    query_cache = load_query_cache(clip_uid)
    qmeta = query_cache["query_metadata"].copy()

    qmeta["clip_uid"] = qmeta["clip_uid"].astype(str)
    qmeta["annotation_uid"] = qmeta["annotation_uid"].astype(str)
    qmeta["qs_id"] = qmeta["qs_id"].astype(int)

    rows = []

    for _, qrow in qmeta.iterrows():
        key = (
            str(qrow["clip_uid"]),
            str(qrow["annotation_uid"]),
            int(qrow["qs_id"]),
        )

        if key not in rt_lookup:
            continue

        rt = rt_lookup[key]

        rt_min = rt["rt_min_frame"]
        rt_max = rt["rt_max_frame"]

        covered_mask = (
            (frame_numbers >= rt_min - tolerance_frames)
            & (frame_numbers <= rt_max + tolerance_frames)
        )

        covered_frames = frame_numbers[covered_mask]

        rows.append({
            "clip_uid": clip_uid,
            "annotation_uid": qrow["annotation_uid"],
            "qs_id": int(qrow["qs_id"]),
            "object_title": qrow["object_title"],
            "rt_min_frame": rt_min,
            "rt_max_frame": rt_max,
            "response_track_len": rt["response_track_len"],
            "num_index_frames": len(frame_numbers),
            "num_gt_covered_index_frames": len(covered_frames),
            "gt_covered_by_index": len(covered_frames) > 0,
            "first_covered_frame": covered_frames[0] if len(covered_frames) > 0 else np.nan,
            "last_covered_frame": covered_frames[-1] if len(covered_frames) > 0 else np.nan,
        })

    return pd.DataFrame(rows)


coverage_rows = []

for clip_uid in tqdm(all_300_clip_uids, desc="Checking GT frame coverage"):
    coverage_rows.append(
        check_gt_frame_coverage_for_one_clip(
            clip_uid=clip_uid,
            tolerance_frames=TOLERANCE_FRAMES,
        )
    )

coverage_df = pd.concat(coverage_rows, ignore_index=True)

print("Coverage rows:", len(coverage_df))
print("Unique clips:", coverage_df["clip_uid"].nunique())
print("GT covered count:", coverage_df["gt_covered_by_index"].sum())
print("GT not covered count:", (~coverage_df["gt_covered_by_index"]).sum())
print("GT coverage rate:", coverage_df["gt_covered_by_index"].mean() * 100)

display(coverage_df.head())

coverage_path = RESULTS_DIR / f"gt_coverage_nclips{N_CLIPS}_tolerance{TOLERANCE_FRAMES}.csv"
coverage_df.to_csv(coverage_path, index=False)

print("Saved GT coverage results to:", coverage_path)

Checking GT frame coverage:   0%|          | 0/300 [00:00<?, ?it/s]

Coverage rows: 1157
Unique clips: 300
GT covered count: 775
GT not covered count: 382
GT coverage rate: 66.98357821953329


,clip_uid,annotation_uid,qs_id,object_title,rt_min_frame,rt_max_frame,response_track_len,num_index_frames,num_gt_covered_index_frames,gt_covered_by_index,first_covered_frame,last_covered_frame
0,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,2,portable fan,42288.0,42317.0,30.0,300,1,True,42300.0,42300.0
1,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,3,tube,46764.0,46770.0,7.0,300,1,True,46770.0,46770.0
2,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,1,green bowl,49488.0,49536.0,49.0,300,0,False,NaN,NaN
3,dfe962ab-6aa7-4888-9378-796817a30ab6,dc29a4cf-1430-43fe-98d2-156ca2b303cf,2,stool,49494.0,49511.0,18.0,300,0,False,NaN,NaN
4,dfe962ab-6aa7-4888-9378-796817a30ab6,dc29a4cf-1430-43fe-98d2-156ca2b303cf,3,bucket,42678.0,42690.0,13.0,300,1,True,42690.0,42690.0


Saved GT coverage results to: /home/jupyter/results/retrieval_subset/gt_coverage_nclips300_tolerance5.csv


## Ground-truth Coverage Upper Bound

We checked whether each query's ground-truth response-track window is covered by at least one sampled index frame in the cached frame set.

### Result

| Metric | Value |
|---|---:|
| Total query cases | 1,157 |
| Unique clips | 300 |
| GT covered by sampled index frames | 775 |
| GT not covered | 382 |
| GT coverage rate | 66.98% |

### Interpretation

The coverage upper bound is substantially higher than the CLIP-only and BLIP-only Top-100 retrieval performance. While CLIP and BLIP achieve only around 25% Top-100 success, approximately 67% of query cases have at least one ground-truth response-track frame present in the sampled index frame set.

This means the low baseline performance is not only caused by missing ground-truth frames in the sampled index. In many cases, the correct frame is available in the index cache but is not ranked highly enough by the current embedding similarity score.

Therefore, tuning strategies such as CLIP/BLIP score fusion, temporal smoothing, and candidate reranking are meaningful next steps.

However, around 33% of query cases are not covered by the sampled index frames. These cases cannot be solved by reranking alone unless we increase frame sampling density or introduce a second-stage search around candidate regions.

In [ ]:
# ---------------- 20. Define CLIP/BLIP alpha fusion retrieval ----------------

def evaluate_one_clip_alpha_fusion(
    clip_uid,
    alpha=0.5,
    topk=100,
    tolerance_frames=TOLERANCE_FRAMES,
):
    """
    Evaluate retrieval for one clip using alpha fusion:

        fusion_score = alpha * CLIP_score + (1 - alpha) * BLIP_score

    alpha = 1.0 -> CLIP-only
    alpha = 0.0 -> BLIP-only
    """
    index_cache = load_index_cache(clip_uid)
    query_cache = load_query_cache(clip_uid)

    frame_numbers = index_cache["frame_numbers"]

    index_clip_embs = index_cache["clip_embs"]
    index_blip_embs = index_cache["blip_embs"]

    query_clip_embs = query_cache["clip_embs"]
    query_blip_embs = query_cache["blip_embs"]

    qmeta = query_cache["query_metadata"].copy()
    qmeta["clip_uid"] = qmeta["clip_uid"].astype(str)
    qmeta["annotation_uid"] = qmeta["annotation_uid"].astype(str)
    qmeta["qs_id"] = qmeta["qs_id"].astype(int)

    rows = []

    for q_idx, qrow in qmeta.iterrows():
        key = (
            str(qrow["clip_uid"]),
            str(qrow["annotation_uid"]),
            int(qrow["qs_id"]),
        )

        if key not in rt_lookup:
            continue

        rt = rt_lookup[key]

        clip_scores = compute_similarity_scores(index_clip_embs, query_clip_embs[q_idx])
        blip_scores = compute_similarity_scores(index_blip_embs, query_blip_embs[q_idx])

        fusion_scores = alpha * clip_scores + (1 - alpha) * blip_scores

        topk_df = get_topk_frames(frame_numbers, fusion_scores, k=topk)

        rank = first_correct_rank(
            topk_df=topk_df,
            rt_min_frame=rt["rt_min_frame"],
            rt_max_frame=rt["rt_max_frame"],
            tolerance_frames=tolerance_frames,
        )

        top1_frame = int(topk_df.iloc[0]["frame_number"])
        top1_score = float(topk_df.iloc[0]["score"])

        rows.append({
            "clip_uid": clip_uid,
            "annotation_uid": qrow["annotation_uid"],
            "qs_id": int(qrow["qs_id"]),
            "object_title": qrow["object_title"],
            "method": "alpha_fusion",
            "alpha": alpha,
            "top1_frame": top1_frame,
            "top1_score": top1_score,
            "rt_min_frame": rt["rt_min_frame"],
            "rt_max_frame": rt["rt_max_frame"],
            "first_correct_rank": rank,
            "top1_success": bool(rank == 1),
            "top5_success": bool(pd.notna(rank) and rank <= 5),
            "top10_success": bool(pd.notna(rank) and rank <= 10),
            "top20_success": bool(pd.notna(rank) and rank <= 20),
            "top50_success": bool(pd.notna(rank) and rank <= 50),
            "top100_success": bool(pd.notna(rank) and rank <= 100),
        })

    return pd.DataFrame(rows)


print("Alpha fusion function defined.")

Alpha fusion function defined.


In [ ]:
# ---------------- 21. One-clip alpha fusion sanity check ----------------

test_clip_uid = all_300_clip_uids[0]

fusion_test_df = evaluate_one_clip_alpha_fusion(
    clip_uid=test_clip_uid,
    alpha=0.5,
    topk=100,
)

print("Test clip:", test_clip_uid)
print("Rows:", len(fusion_test_df))
display(fusion_test_df)

Test clip: dfe962ab-6aa7-4888-9378-796817a30ab6
Rows: 6


,clip_uid,annotation_uid,qs_id,object_title,method,alpha,top1_frame,top1_score,rt_min_frame,rt_max_frame,first_correct_rank,top1_success,top5_success,top10_success,top20_success,top50_success,top100_success
0,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,2,portable fan,alpha_fusion,0.5,42390,0.774597,42288.0,42317.0,71.0,False,False,False,False,False,True
1,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,3,tube,alpha_fusion,0.5,41370,0.708696,46764.0,46770.0,3.0,False,True,True,True,True,True
2,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,1,green bowl,alpha_fusion,0.5,41370,0.714960,49488.0,49536.0,NaN,False,False,False,False,False,False
3,dfe962ab-6aa7-4888-9378-796817a30ab6,dc29a4cf-1430-43fe-98d2-156ca2b303cf,2,stool,alpha_fusion,0.5,46770,0.777815,49494.0,49511.0,NaN,False,False,False,False,False,False
4,dfe962ab-6aa7-4888-9378-796817a30ab6,dc29a4cf-1430-43fe-98d2-156ca2b303cf,3,bucket,alpha_fusion,0.5,46770,0.786738,42678.0,42690.0,34.0,False,False,False,False,True,True
5,dfe962ab-6aa7-4888-9378-796817a30ab6,dc29a4cf-1430-43fe-98d2-156ca2b303cf,1,stool,alpha_fusion,0.5,42480,0.780019,43434.0,43441.0,83.0,False,False,False,False,False,True


In [ ]:
# ---------------- 22. Run alpha fusion grid over 300 clips ----------------

alphas = [0.0, 0.25, 0.5, 0.75, 1.0]

fusion_rows = []

for alpha in alphas:
    print("=" * 80)
    print("Running alpha fusion:", alpha)

    for clip_uid in tqdm(all_300_clip_uids, desc=f"alpha={alpha}"):
        df_one = evaluate_one_clip_alpha_fusion(
            clip_uid=clip_uid,
            alpha=alpha,
            topk=100,
        )
        fusion_rows.append(df_one)

fusion_df = pd.concat(fusion_rows, ignore_index=True)

print("Fusion rows:", len(fusion_df))
print("Alpha counts:")
print(fusion_df["alpha"].value_counts().sort_index())

display(fusion_df.head())

fusion_path = RESULTS_DIR / f"alpha_fusion_nclips{N_CLIPS}_top100.csv"
fusion_df.to_csv(fusion_path, index=False)

print("Saved alpha fusion results to:", fusion_path)

Running alpha fusion: 0.0


alpha=0.0:   0%|          | 0/300 [00:00<?, ?it/s]

Running alpha fusion: 0.25


alpha=0.25:   0%|          | 0/300 [00:00<?, ?it/s]

Running alpha fusion: 0.5


...

Running alpha fusion: 1.0


alpha=1.0:   0%|          | 0/300 [00:00<?, ?it/s]

Fusion rows: 5785
Alpha counts:
alpha
0.00    1157
0.25    1157
0.50    1157
0.75    1157
1.00    1157
Name: count, dtype: int64


,clip_uid,annotation_uid,qs_id,object_title,method,alpha,top1_frame,top1_score,rt_min_frame,rt_max_frame,first_correct_rank,top1_success,top5_success,top10_success,top20_success,top50_success,top100_success
0,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,2,portable fan,alpha_fusion,0.0,42390,0.753312,42288.0,42317.0,73.0,False,False,False,False,False,True
1,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,3,tube,alpha_fusion,0.0,41370,0.622319,46764.0,46770.0,3.0,False,True,True,True,True,True
2,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,1,green bowl,alpha_fusion,0.0,47670,0.650754,49488.0,49536.0,NaN,False,False,False,False,False,False
3,dfe962ab-6aa7-4888-9378-796817a30ab6,dc29a4cf-1430-43fe-98d2-156ca2b303cf,2,stool,alpha_fusion,0.0,46770,0.703079,49494.0,49511.0,NaN,False,False,False,False,False,False
4,dfe962ab-6aa7-4888-9378-796817a30ab6,dc29a4cf-1430-43fe-98d2-156ca2b303cf,3,bucket,alpha_fusion,0.0,42300,0.721368,42678.0,42690.0,47.0,False,False,False,False,True,True


Saved alpha fusion results to: /home/jupyter/results/retrieval_subset/alpha_fusion_nclips300_top100.csv


In [ ]:
# ---------------- 23. Summarize alpha fusion results ----------------

metric_cols = [
    "top1_success",
    "top5_success",
    "top10_success",
    "top20_success",
    "top50_success",
    "top100_success",
]

fusion_summary = (
    fusion_df
    .groupby("alpha")[metric_cols]
    .mean()
    .reset_index()
)

for col in metric_cols:
    fusion_summary[col] = fusion_summary[col] * 100

display(fusion_summary)

fusion_summary_path = RESULTS_DIR / f"alpha_fusion_summary_nclips{N_CLIPS}_top100.csv"
fusion_summary.to_csv(fusion_summary_path, index=False)

print("Saved alpha fusion summary to:", fusion_summary_path)

,alpha,top1_success,top5_success,top10_success,top20_success,top50_success,top100_success
0,0.00,0.518583,1.469317,2.852204,5.617978,13.569576,25.669836
1,0.25,0.518583,1.555748,2.852204,5.185825,13.742437,25.151253
2,0.50,0.605013,1.728608,2.765774,5.445117,13.828868,25.842697
3,0.75,0.432152,1.901469,3.025065,5.445117,14.174589,25.496975
4,1.00,0.518583,1.728608,2.938634,5.272256,15.038894,24.978392


Saved alpha fusion summary to: /home/jupyter/results/retrieval_subset/alpha_fusion_summary_nclips300_top100.csv


Simple CLIP/BLIP alpha fusion provides only marginal improvement over single-model baselines.
The best early-rank setting is alpha=0.75, suggesting that CLIP should dominate the fusion score while BLIP provides auxiliary signal.
However, the improvement is small, so later experiments should focus more on temporal/candidate reranking rather than relying only on linear score fusion.

In [ ]:
## CLIP/BLIP Alpha Fusion Results

We evaluated simple linear score fusion between CLIP and BLIP embeddings:

`fusion_score = alpha * CLIP_score + (1 - alpha) * BLIP_score`

### Result

| Alpha | Interpretation | Top-1 | Top-5 | Top-10 | Top-20 | Top-50 | Top-100 |
|---:|---|---:|---:|---:|---:|---:|---:|
| 0.00 | BLIP-only | 0.52% | 1.47% | 2.85% | 5.62% | 13.57% | 25.67% |
| 0.25 | 25% CLIP + 75% BLIP | 0.52% | 1.56% | 2.85% | 5.19% | 13.74% | 25.15% |
| 0.50 | 50% CLIP + 50% BLIP | 0.61% | 1.73% | 2.77% | 5.45% | 13.83% | 25.84% |
| 0.75 | 75% CLIP + 25% BLIP | 0.43% | 1.90% | 3.03% | 5.45% | 14.17% | 25.50% |
| 1.00 | CLIP-only | 0.52% | 1.73% | 2.94% | 5.27% | 15.04% | 24.98% |

### Interpretation

Simple CLIP/BLIP fusion provides only marginal improvement over single-model baselines. The best Top-100 result is achieved by alpha = 0.50, but the gain over BLIP-only is very small. For early-rank retrieval, alpha = 0.75 performs best at Top-5 and Top-10, suggesting that CLIP should remain the dominant signal while BLIP provides auxiliary information.

Overall, linear fusion alone is not sufficient to substantially improve retrieval. Since the ground-truth coverage upper bound is around 67%, while Top-100 retrieval remains around 25%, the next improvement should focus on temporal smoothing or candidate-guided reranking rather than simple score fusion alone.

In [ ]:
# ---------------- 24. Define temporal smoothing helper ----------------

def smooth_scores(scores, window_size=5, mode="mean"):
    """
    Smooth similarity scores across neighboring sampled index frames.

    window_size:
        Number of neighboring index positions included in smoothing.
        Should be an odd number, e.g. 3, 5, 7.

    mode:
        - "mean": average score in the window
        - "max": max score in the window
    """
    scores = np.asarray(scores, dtype=float)
    n = len(scores)

    if window_size <= 1:
        return scores.copy()

    if window_size % 2 == 0:
        raise ValueError("window_size should be odd, e.g. 3, 5, 7.")

    half = window_size // 2
    smoothed = np.zeros_like(scores)

    for i in range(n):
        start = max(0, i - half)
        end = min(n, i + half + 1)
        window_scores = scores[start:end]

        if mode == "mean":
            smoothed[i] = window_scores.mean()
        elif mode == "max":
            smoothed[i] = window_scores.max()
        else:
            raise ValueError("mode must be 'mean' or 'max'.")

    return smoothed


print("Temporal smoothing helper defined.")

Temporal smoothing helper defined.


In [ ]:
# ---------------- 25. Define one-clip temporal smoothing evaluation ----------------

def evaluate_one_clip_temporal_smoothing(
    clip_uid,
    alpha=0.75,
    window_size=5,
    smoothing_mode="mean",
    topk=100,
    tolerance_frames=TOLERANCE_FRAMES,
):
    """
    Evaluate alpha-fusion retrieval after temporal smoothing.

    Steps:
    1. Compute CLIP score
    2. Compute BLIP score
    3. Fuse scores
    4. Smooth fused scores across neighboring sampled index frames
    5. Rank frames by smoothed score
    """
    index_cache = load_index_cache(clip_uid)
    query_cache = load_query_cache(clip_uid)

    frame_numbers = index_cache["frame_numbers"]

    index_clip_embs = index_cache["clip_embs"]
    index_blip_embs = index_cache["blip_embs"]

    query_clip_embs = query_cache["clip_embs"]
    query_blip_embs = query_cache["blip_embs"]

    qmeta = query_cache["query_metadata"].copy()
    qmeta["clip_uid"] = qmeta["clip_uid"].astype(str)
    qmeta["annotation_uid"] = qmeta["annotation_uid"].astype(str)
    qmeta["qs_id"] = qmeta["qs_id"].astype(int)

    rows = []

    for q_idx, qrow in qmeta.iterrows():
        key = (
            str(qrow["clip_uid"]),
            str(qrow["annotation_uid"]),
            int(qrow["qs_id"]),
        )

        if key not in rt_lookup:
            continue

        rt = rt_lookup[key]

        clip_scores = compute_similarity_scores(index_clip_embs, query_clip_embs[q_idx])
        blip_scores = compute_similarity_scores(index_blip_embs, query_blip_embs[q_idx])

        fusion_scores = alpha * clip_scores + (1 - alpha) * blip_scores
        smoothed_scores = smooth_scores(
            fusion_scores,
            window_size=window_size,
            mode=smoothing_mode,
        )

        topk_df = get_topk_frames(frame_numbers, smoothed_scores, k=topk)

        rank = first_correct_rank(
            topk_df=topk_df,
            rt_min_frame=rt["rt_min_frame"],
            rt_max_frame=rt["rt_max_frame"],
            tolerance_frames=tolerance_frames,
        )

        rows.append({
            "clip_uid": clip_uid,
            "annotation_uid": qrow["annotation_uid"],
            "qs_id": int(qrow["qs_id"]),
            "object_title": qrow["object_title"],
            "method": "temporal_smoothing",
            "alpha": alpha,
            "window_size": window_size,
            "smoothing_mode": smoothing_mode,
            "top1_frame": int(topk_df.iloc[0]["frame_number"]),
            "top1_score": float(topk_df.iloc[0]["score"]),
            "rt_min_frame": rt["rt_min_frame"],
            "rt_max_frame": rt["rt_max_frame"],
            "first_correct_rank": rank,
            "top1_success": bool(rank == 1),
            "top5_success": bool(pd.notna(rank) and rank <= 5),
            "top10_success": bool(pd.notna(rank) and rank <= 10),
            "top20_success": bool(pd.notna(rank) and rank <= 20),
            "top50_success": bool(pd.notna(rank) and rank <= 50),
            "top100_success": bool(pd.notna(rank) and rank <= 100),
        })

    return pd.DataFrame(rows)


print("Temporal smoothing evaluation function defined.")

Temporal smoothing evaluation function defined.


In [ ]:
# ---------------- 26. One-clip temporal smoothing sanity check ----------------

test_clip_uid = all_300_clip_uids[0]

smooth_test_df = evaluate_one_clip_temporal_smoothing(
    clip_uid=test_clip_uid,
    alpha=0.75,
    window_size=5,
    smoothing_mode="mean",
    topk=100,
)

print("Test clip:", test_clip_uid)
print("Rows:", len(smooth_test_df))
display(smooth_test_df)

Test clip: dfe962ab-6aa7-4888-9378-796817a30ab6
Rows: 6


,clip_uid,annotation_uid,qs_id,object_title,method,alpha,window_size,smoothing_mode,top1_frame,top1_score,rt_min_frame,rt_max_frame,first_correct_rank,top1_success,top5_success,top10_success,top20_success,top50_success,top100_success
0,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,2,portable fan,temporal_smoothing,0.75,5,mean,43770,0.753529,42288.0,42317.0,78.0,False,False,False,False,False,True
1,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,3,tube,temporal_smoothing,0.75,5,mean,43440,0.634601,46764.0,46770.0,NaN,False,False,False,False,False,False
2,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,1,green bowl,temporal_smoothing,0.75,5,mean,43440,0.664668,49488.0,49536.0,NaN,False,False,False,False,False,False
3,dfe962ab-6aa7-4888-9378-796817a30ab6,dc29a4cf-1430-43fe-98d2-156ca2b303cf,2,stool,temporal_smoothing,0.75,5,mean,44430,0.742072,49494.0,49511.0,NaN,False,False,False,False,False,False
4,dfe962ab-6aa7-4888-9378-796817a30ab6,dc29a4cf-1430-43fe-98d2-156ca2b303cf,3,bucket,temporal_smoothing,0.75,5,mean,43440,0.759456,42678.0,42690.0,NaN,False,False,False,False,False,False
5,dfe962ab-6aa7-4888-9378-796817a30ab6,dc29a4cf-1430-43fe-98d2-156ca2b303cf,1,stool,temporal_smoothing,0.75,5,mean,44460,0.740068,43434.0,43441.0,11.0,False,False,False,True,True,True


In [ ]:
# ---------------- 27. Run temporal smoothing grid over 300 clips ----------------

smoothing_grid = []

alphas = [0.5, 0.75, 1.0]
window_sizes = [3, 5, 7]
smoothing_modes = ["mean", "max"]

for alpha in alphas:
    for window_size in window_sizes:
        for smoothing_mode in smoothing_modes:
            smoothing_grid.append({
                "alpha": alpha,
                "window_size": window_size,
                "smoothing_mode": smoothing_mode,
            })

print("Total smoothing settings:", len(smoothing_grid))
display(pd.DataFrame(smoothing_grid))

Total smoothing settings: 18


,alpha,window_size,smoothing_mode
0,0.50,3,mean
1,0.50,3,max
2,0.50,5,mean
3,0.50,5,max
4,0.50,7,mean
5,0.50,7,max
6,0.75,3,mean
7,0.75,3,max
8,0.75,5,mean
9,0.75,5,max


In [ ]:
# ---------------- 28. Full temporal smoothing grid evaluation ----------------

smooth_rows = []

for cfg in smoothing_grid:
    alpha = cfg["alpha"]
    window_size = cfg["window_size"]
    smoothing_mode = cfg["smoothing_mode"]

    print("=" * 100)
    print(f"Running temporal smoothing | alpha={alpha} | window={window_size} | mode={smoothing_mode}")

    for clip_uid in tqdm(
        all_300_clip_uids,
        desc=f"a={alpha}, w={window_size}, {smoothing_mode}"
    ):
        df_one = evaluate_one_clip_temporal_smoothing(
            clip_uid=clip_uid,
            alpha=alpha,
            window_size=window_size,
            smoothing_mode=smoothing_mode,
            topk=100,
        )
        smooth_rows.append(df_one)

smooth_df = pd.concat(smooth_rows, ignore_index=True)

print("Temporal smoothing rows:", len(smooth_df))
print("Expected rows:", 1157 * len(smoothing_grid))

display(smooth_df.head())

smooth_path = RESULTS_DIR / f"temporal_smoothing_nclips{N_CLIPS}_top100.csv"
smooth_df.to_csv(smooth_path, index=False)

print("Saved temporal smoothing results to:", smooth_path)

Running temporal smoothing | alpha=0.5 | window=3 | mode=mean


a=0.5, w=3, mean:   0%|          | 0/300 [00:00<?, ?it/s]

Running temporal smoothing | alpha=0.5 | window=3 | mode=max


a=0.5, w=3, max:   0%|          | 0/300 [00:00<?, ?it/s]

Running temporal smoothing | alpha=0.5 | window=5 | mode=mean


...

Running temporal smoothing | alpha=1.0 | window=7 | mode=max


a=1.0, w=7, max:   0%|          | 0/300 [00:00<?, ?it/s]

Temporal smoothing rows: 20826
Expected rows: 20826


,clip_uid,annotation_uid,qs_id,object_title,method,alpha,window_size,smoothing_mode,top1_frame,top1_score,rt_min_frame,rt_max_frame,first_correct_rank,top1_success,top5_success,top10_success,top20_success,top50_success,top100_success
0,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,2,portable fan,temporal_smoothing,0.5,3,mean,42420,0.745986,42288.0,42317.0,74.0,False,False,False,False,False,True
1,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,3,tube,temporal_smoothing,0.5,3,mean,41370,0.622939,46764.0,46770.0,37.0,False,False,False,False,True,True
2,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,1,green bowl,temporal_smoothing,0.5,3,mean,41370,0.654436,49488.0,49536.0,NaN,False,False,False,False,False,False
3,dfe962ab-6aa7-4888-9378-796817a30ab6,dc29a4cf-1430-43fe-98d2-156ca2b303cf,2,stool,temporal_smoothing,0.5,3,mean,43440,0.711185,49494.0,49511.0,NaN,False,False,False,False,False,False
4,dfe962ab-6aa7-4888-9378-796817a30ab6,dc29a4cf-1430-43fe-98d2-156ca2b303cf,3,bucket,temporal_smoothing,0.5,3,mean,41340,0.733470,42678.0,42690.0,70.0,False,False,False,False,False,True


Saved temporal smoothing results to: /home/jupyter/results/retrieval_subset/temporal_smoothing_nclips300_top100.csv


In [ ]:
# ---------------- 29. Summarize temporal smoothing results ----------------

metric_cols = [
    "top1_success",
    "top5_success",
    "top10_success",
    "top20_success",
    "top50_success",
    "top100_success",
]

smooth_summary = (
    smooth_df
    .groupby(["alpha", "window_size", "smoothing_mode"])[metric_cols]
    .mean()
    .reset_index()
)

for col in metric_cols:
    smooth_summary[col] = smooth_summary[col] * 100

smooth_summary_sorted = smooth_summary.sort_values(
    ["top100_success", "top50_success", "top20_success", "top10_success"],
    ascending=False,
).reset_index(drop=True)

display(smooth_summary_sorted)

smooth_summary_path = RESULTS_DIR / f"temporal_smoothing_summary_nclips{N_CLIPS}_top100.csv"
smooth_summary_sorted.to_csv(smooth_summary_path, index=False)

print("Saved temporal smoothing summary to:", smooth_summary_path)

,alpha,window_size,smoothing_mode,top1_success,top5_success,top10_success,top20_success,top50_success,top100_success
0,0.50,7,max,0.777874,1.987900,3.630078,7.000864,14.606742,25.756266
1,0.50,3,max,0.432152,2.074330,4.062230,5.790838,13.396716,25.756266
2,0.75,3,max,0.172861,1.987900,4.321521,6.222990,13.396716,25.151253
3,0.75,5,max,0.259291,1.210026,2.938634,6.309421,12.705272,25.064823
4,0.50,5,max,0.518583,1.901469,3.889369,6.482282,13.137424,24.891962
5,0.75,7,max,0.172861,1.123596,3.284356,6.568712,13.569576,24.805532
6,1.00,3,max,0.172861,1.901469,3.457217,5.704408,13.310285,24.805532
7,0.50,3,mean,0.000000,1.210026,3.025065,5.012965,12.273120,24.805532
8,1.00,5,max,0.345722,1.382887,2.765774,5.531547,12.791703,24.459810
9,0.50,5,mean,0.259291,0.950735,2.247191,5.445117,11.668107,24.286949


Saved temporal smoothing summary to: /home/jupyter/results/retrieval_subset/temporal_smoothing_summary_nclips300_top100.csv


## Temporal Smoothing Results

We evaluated temporal smoothing over fused CLIP/BLIP similarity scores using different alpha values, window sizes, and smoothing modes.

The best Top-100 performance is achieved by max smoothing with alpha = 0.50 and window size = 3 or 7, reaching approximately 25.76% Top-100 accuracy. This is close to, but does not exceed, the best alpha-fusion result of 25.84%.

However, temporal smoothing improves early-rank retrieval. The best Top-20 result increases to 7.00%, compared with around 5.45% from alpha fusion. This suggests that temporal smoothing helps promote correct frames when they are already near strong candidates, but it does not substantially improve overall candidate recall.

Max smoothing consistently performs better than mean smoothing. This likely happens because response tracks can be short, and averaging across neighboring frames may dilute sharp local similarity peaks. Max smoothing preserves these local peaks better.

Overall, temporal smoothing is useful as a lightweight reranking strategy, especially for improving Top-10 and Top-20 performance, but it does not solve the broader recall gap between the 66.98% ground-truth coverage upper bound and the roughly 25% Top-100 retrieval performance.

In [ ]:
# ---------------- 30 fixed. Candidate-guided window reranking ----------------

def evaluate_one_clip_candidate_window(
    clip_uid,
    alpha=0.5,
    candidate_topk=20,
    window_radius=2,
    score_mode="max",
    final_topk=100,
    tolerance_frames=TOLERANCE_FRAMES,
):
    """
    Candidate-guided window reranking.

    Steps:
    1. Compute CLIP and BLIP similarity scores.
    2. Fuse scores.
    3. Select top candidate_topk frames as anchors.
    4. Expand around each anchor by +/- window_radius index positions.
    5. Aggregate scores for frames in candidate windows.
    6. Rank expanded candidate frames by aggregated score.
    """

    index_cache = load_index_cache(clip_uid)
    query_cache = load_query_cache(clip_uid)

    frame_numbers = index_cache["frame_numbers"]

    index_clip_embs = index_cache["clip_embs"]
    index_blip_embs = index_cache["blip_embs"]

    query_clip_embs = query_cache["clip_embs"]
    query_blip_embs = query_cache["blip_embs"]

    qmeta = query_cache["query_metadata"].copy()
    qmeta["clip_uid"] = qmeta["clip_uid"].astype(str)
    qmeta["annotation_uid"] = qmeta["annotation_uid"].astype(str)
    qmeta["qs_id"] = qmeta["qs_id"].astype(int)

    rows = []

    for q_idx, qrow in qmeta.iterrows():
        key = (
            str(qrow["clip_uid"]),
            str(qrow["annotation_uid"]),
            int(qrow["qs_id"]),
        )

        if key not in rt_lookup:
            continue

        rt = rt_lookup[key]

        clip_scores = compute_similarity_scores(index_clip_embs, query_clip_embs[q_idx])
        blip_scores = compute_similarity_scores(index_blip_embs, query_blip_embs[q_idx])

        fusion_scores = alpha * clip_scores + (1 - alpha) * blip_scores

        candidate_topk_actual = min(candidate_topk, len(fusion_scores))
        anchor_indices = np.argsort(-fusion_scores)[:candidate_topk_actual]

        expanded_scores = {}

        for anchor_idx in anchor_indices:
            start = max(0, anchor_idx - window_radius)
            end = min(len(fusion_scores), anchor_idx + window_radius + 1)

            for idx in range(start, end):
                if idx not in expanded_scores:
                    expanded_scores[idx] = []
                expanded_scores[idx].append(float(fusion_scores[idx]))

        rerank_rows = []

        for idx, score_list in expanded_scores.items():
            if score_mode == "max":
                agg_score = max(score_list)
            elif score_mode == "mean":
                agg_score = float(np.mean(score_list))
            else:
                raise ValueError("score_mode must be 'max' or 'mean'.")

            rerank_rows.append({
                "frame_number": int(frame_numbers[idx]),
                "score": agg_score,
                "index_position": int(idx),
            })

        topk_df = (
            pd.DataFrame(rerank_rows)
            .sort_values("score", ascending=False)
            .head(final_topk)
            .reset_index(drop=True)
        )

        # Important: first_correct_rank() expects a rank column
        topk_df["rank"] = np.arange(1, len(topk_df) + 1)

        rank = first_correct_rank(
            topk_df=topk_df,
            rt_min_frame=rt["rt_min_frame"],
            rt_max_frame=rt["rt_max_frame"],
            tolerance_frames=tolerance_frames,
        )

        rows.append({
            "clip_uid": clip_uid,
            "annotation_uid": qrow["annotation_uid"],
            "qs_id": int(qrow["qs_id"]),
            "object_title": qrow["object_title"],
            "method": "candidate_window",
            "alpha": alpha,
            "candidate_topk": candidate_topk,
            "window_radius": window_radius,
            "score_mode": score_mode,
            "num_expanded_frames": len(topk_df),
            "top1_frame": int(topk_df.iloc[0]["frame_number"]),
            "top1_score": float(topk_df.iloc[0]["score"]),
            "rt_min_frame": rt["rt_min_frame"],
            "rt_max_frame": rt["rt_max_frame"],
            "first_correct_rank": rank,
            "top1_success": bool(rank == 1),
            "top5_success": bool(pd.notna(rank) and rank <= 5),
            "top10_success": bool(pd.notna(rank) and rank <= 10),
            "top20_success": bool(pd.notna(rank) and rank <= 20),
            "top50_success": bool(pd.notna(rank) and rank <= 50),
            "top100_success": bool(pd.notna(rank) and rank <= 100),
        })

    return pd.DataFrame(rows)


print("Fixed candidate-guided window reranking function defined.")

Fixed candidate-guided window reranking function defined.


In [ ]:
# ---------------- 31. One-clip candidate-window sanity check ----------------

test_clip_uid = all_300_clip_uids[0]

candidate_test_df = evaluate_one_clip_candidate_window(
    clip_uid=test_clip_uid,
    alpha=0.5,
    candidate_topk=20,
    window_radius=2,
    score_mode="max",
    final_topk=100,
)

print("Test clip:", test_clip_uid)
print("Rows:", len(candidate_test_df))
display(candidate_test_df)

Test clip: dfe962ab-6aa7-4888-9378-796817a30ab6
Rows: 6


,clip_uid,annotation_uid,qs_id,object_title,method,alpha,candidate_topk,window_radius,score_mode,num_expanded_frames,...,top1_score,rt_min_frame,rt_max_frame,first_correct_rank,top1_success,top5_success,top10_success,top20_success,top50_success,top100_success
0,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,2,portable fan,candidate_window,0.5,20,2,max,41,...,0.774597,42288.0,42317.0,NaN,False,False,False,False,False,False
1,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,3,tube,candidate_window,0.5,20,2,max,65,...,0.708696,46764.0,46770.0,3.0,False,True,True,True,True,True
2,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,1,green bowl,candidate_window,0.5,20,2,max,78,...,0.714960,49488.0,49536.0,NaN,False,False,False,False,False,False
3,dfe962ab-6aa7-4888-9378-796817a30ab6,dc29a4cf-1430-43fe-98d2-156ca2b303cf,2,stool,candidate_window,0.5,20,2,max,73,...,0.777815,49494.0,49511.0,NaN,False,False,False,False,False,False
4,dfe962ab-6aa7-4888-9378-796817a30ab6,dc29a4cf-1430-43fe-98d2-156ca2b303cf,3,bucket,candidate_window,0.5,20,2,max,78,...,0.786738,42678.0,42690.0,NaN,False,False,False,False,False,False
5,dfe962ab-6aa7-4888-9378-796817a30ab6,dc29a4cf-1430-43fe-98d2-156ca2b303cf,1,stool,candidate_window,0.5,20,2,max,67,...,0.780019,43434.0,43441.0,42.0,False,False,False,False,True,True


In [ ]:
# ---------------- 32. Define candidate-window grid ----------------

candidate_grid = []

alphas = [0.5, 0.75]
candidate_topks = [20, 50, 100]
window_radii = [1, 2, 3, 5]
score_modes = ["max"]

for alpha in alphas:
    for candidate_topk in candidate_topks:
        for window_radius in window_radii:
            for score_mode in score_modes:
                candidate_grid.append({
                    "alpha": alpha,
                    "candidate_topk": candidate_topk,
                    "window_radius": window_radius,
                    "score_mode": score_mode,
                })

candidate_grid_df = pd.DataFrame(candidate_grid)

print("Total candidate-window settings:", len(candidate_grid_df))
display(candidate_grid_df)

Total candidate-window settings: 24


,alpha,candidate_topk,window_radius,score_mode
0,0.50,20,1,max
1,0.50,20,2,max
2,0.50,20,3,max
3,0.50,20,5,max
4,0.50,50,1,max
5,0.50,50,2,max
6,0.50,50,3,max
7,0.50,50,5,max
8,0.50,100,1,max
9,0.50,100,2,max


In [ ]:
# ---------------- 33. Run candidate-window grid safely ----------------

CANDIDATE_RESULT_DIR = RESULTS_DIR / "candidate_window_results"
CANDIDATE_RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("Candidate result dir:", CANDIDATE_RESULT_DIR)

all_candidate_result_paths = []

for cfg_idx, cfg in enumerate(candidate_grid):
    alpha = cfg["alpha"]
    candidate_topk = cfg["candidate_topk"]
    window_radius = cfg["window_radius"]
    score_mode = cfg["score_mode"]

    cfg_name = (
        f"candidate_window_"
        f"alpha{alpha}_"
        f"topk{candidate_topk}_"
        f"radius{window_radius}_"
        f"{score_mode}_"
        f"nclips{N_CLIPS}.csv"
    )

    cfg_path = CANDIDATE_RESULT_DIR / cfg_name
    all_candidate_result_paths.append(cfg_path)

    print("=" * 100)
    print(f"[{cfg_idx + 1}/{len(candidate_grid)}] Running:", cfg)
    print("Save path:", cfg_path)

    # Skip if already completed
    if cfg_path.exists():
        existing_df = pd.read_csv(cfg_path)
        if len(existing_df) == 1157:
            print("Already completed. Skipping.")
            continue
        else:
            print("Existing file incomplete. Recomputing:", len(existing_df))

    cfg_rows = []

    for clip_uid in tqdm(
        all_300_clip_uids,
        desc=f"a={alpha}, topk={candidate_topk}, r={window_radius}"
    ):
        df_one = evaluate_one_clip_candidate_window(
            clip_uid=clip_uid,
            alpha=alpha,
            candidate_topk=candidate_topk,
            window_radius=window_radius,
            score_mode=score_mode,
            final_topk=100,
        )
        cfg_rows.append(df_one)

    cfg_df = pd.concat(cfg_rows, ignore_index=True)

    print("Rows:", len(cfg_df))
    cfg_df.to_csv(cfg_path, index=False)
    print("Saved:", cfg_path)

print("All candidate-window config runs finished or skipped.")

Candidate result dir: /home/jupyter/results/retrieval_subset/candidate_window_results
[1/24] Running: {'alpha': 0.5, 'candidate_topk': 20, 'window_radius': 1, 'score_mode': 'max'}
Save path: /home/jupyter/results/retrieval_subset/candidate_window_results/candidate_window_alpha0.5_topk20_radius1_max_nclips300.csv
Already completed. Skipping.
[2/24] Running: {'alpha': 0.5, 'candidate_topk': 20, 'window_radius': 2, 'score_mode': 'max'}
Save path: /home/jupyter/results/retrieval_subset/candidate_window_results/candidate_window_alpha0.5_topk20_radius2_max_nclips300.csv
Already completed. Skipping.
[3/24] Running: {'alpha': 0.5, 'candidate_topk': 20, 'window_radius': 3, 'score_mode': 'max'}
Save path: /home/jupyter/results/retrieval_subset/candidate_window_results/candidate_window_alpha0.5_topk20_radius3_max_nclips300.csv
Already completed. Skipping.
[4/24] Running: {'alpha': 0.5, 'candidate_topk': 20, 'window_radius': 5, 'score_mode': 'max'}
Save path: /home/jupyter/results/retrieval_subset

In [ ]:
# ---------------- 34. Combine candidate-window results and summarize ----------------

candidate_result_files = sorted(CANDIDATE_RESULT_DIR.glob("candidate_window_*.csv"))

print("Candidate result files:", len(candidate_result_files))
for p in candidate_result_files[:5]:
    print(p.name)

candidate_dfs = []

for p in candidate_result_files:
    df = pd.read_csv(p)
    candidate_dfs.append(df)

candidate_df = pd.concat(candidate_dfs, ignore_index=True)

print("Combined candidate-window rows:", len(candidate_df))
print("Expected rows:", 1157 * len(candidate_result_files))

display(candidate_df.head())

candidate_all_path = RESULTS_DIR / f"candidate_window_all_nclips{N_CLIPS}_top100.csv"
candidate_df.to_csv(candidate_all_path, index=False)

print("Saved combined candidate-window results to:", candidate_all_path)


metric_cols = [
    "top1_success",
    "top5_success",
    "top10_success",
    "top20_success",
    "top50_success",
    "top100_success",
]

candidate_summary = (
    candidate_df
    .groupby(["alpha", "candidate_topk", "window_radius", "score_mode"])[metric_cols]
    .mean()
    .reset_index()
)

for col in metric_cols:
    candidate_summary[col] = candidate_summary[col] * 100

candidate_summary_sorted = candidate_summary.sort_values(
    ["top100_success", "top50_success", "top20_success", "top10_success"],
    ascending=False,
).reset_index(drop=True)

display(candidate_summary_sorted)

candidate_summary_path = RESULTS_DIR / f"candidate_window_summary_nclips{N_CLIPS}_top100.csv"
candidate_summary_sorted.to_csv(candidate_summary_path, index=False)

print("Saved candidate-window summary to:", candidate_summary_path)

Candidate result files: 24
candidate_window_alpha0.5_topk100_radius1_max_nclips300.csv
candidate_window_alpha0.5_topk100_radius2_max_nclips300.csv
candidate_window_alpha0.5_topk100_radius3_max_nclips300.csv
candidate_window_alpha0.5_topk100_radius5_max_nclips300.csv
candidate_window_alpha0.5_topk20_radius1_max_nclips300.csv
Combined candidate-window rows: 27768
Expected rows: 27768


,clip_uid,annotation_uid,qs_id,object_title,method,alpha,candidate_topk,window_radius,score_mode,num_expanded_frames,...,top1_score,rt_min_frame,rt_max_frame,first_correct_rank,top1_success,top5_success,top10_success,top20_success,top50_success,top100_success
0,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,2,portable fan,candidate_window,0.5,100,1,max,100,...,0.774597,42288.0,42317.0,71.0,False,False,False,False,False,True
1,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,3,tube,candidate_window,0.5,100,1,max,100,...,0.708696,46764.0,46770.0,3.0,False,True,True,True,True,True
2,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,1,green bowl,candidate_window,0.5,100,1,max,100,...,0.714960,49488.0,49536.0,NaN,False,False,False,False,False,False
3,dfe962ab-6aa7-4888-9378-796817a30ab6,dc29a4cf-1430-43fe-98d2-156ca2b303cf,2,stool,candidate_window,0.5,100,1,max,100,...,0.777815,49494.0,49511.0,NaN,False,False,False,False,False,False
4,dfe962ab-6aa7-4888-9378-796817a30ab6,dc29a4cf-1430-43fe-98d2-156ca2b303cf,3,bucket,candidate_window,0.5,100,1,max,100,...,0.786738,42678.0,42690.0,34.0,False,False,False,False,True,True


Saved combined candidate-window results to: /home/jupyter/results/retrieval_subset/candidate_window_all_nclips300_top100.csv


,alpha,candidate_topk,window_radius,score_mode,top1_success,top5_success,top10_success,top20_success,top50_success,top100_success
0,0.50,50,5,max,0.605013,1.728608,2.765774,5.445117,13.828868,26.620570
1,0.75,50,5,max,0.432152,1.901469,3.025065,5.445117,14.174589,26.274849
2,0.50,100,1,max,0.605013,1.728608,2.765774,5.445117,13.828868,25.842697
3,0.50,100,2,max,0.605013,1.728608,2.765774,5.445117,13.828868,25.842697
4,0.50,100,3,max,0.605013,1.728608,2.765774,5.445117,13.828868,25.842697
5,0.50,100,5,max,0.605013,1.728608,2.765774,5.445117,13.828868,25.842697
6,0.50,20,5,max,0.605013,1.728608,2.765774,5.445117,14.520311,25.583405
7,0.75,100,1,max,0.432152,1.901469,3.025065,5.445117,14.174589,25.496975
8,0.75,100,2,max,0.432152,1.901469,3.025065,5.445117,14.174589,25.496975
9,0.75,100,3,max,0.432152,1.901469,3.025065,5.445117,14.174589,25.496975


Saved candidate-window summary to: /home/jupyter/results/retrieval_subset/candidate_window_summary_nclips300_top100.csv


## Candidate-Guided Window Reranking Results

We evaluated candidate-guided window reranking over 24 configurations using different CLIP/BLIP fusion weights, candidate anchor sizes, and temporal window radii.

The best Top-100 performance was achieved with:

- alpha = 0.50
- candidate_topk = 50
- window_radius = 5
- score_mode = max

This configuration achieved approximately 26.62% Top-100 accuracy, which improves over the previous best alpha-fusion result of 25.84% and the best temporal smoothing result of 25.76%.

However, candidate-window reranking did not improve early-rank retrieval as much as temporal smoothing. Its Top-20 accuracy remains around 5.45%, while temporal smoothing reached approximately 7.00% Top-20 accuracy. This suggests that candidate-window expansion is more useful for improving candidate recall than for pushing correct frames into the very top ranks.

Overall, candidate-guided window reranking is the best current method for Top-100 retrieval, while temporal smoothing remains useful for early-rank improvement.

In [ ]:
# ---------------- 35. Final comparison across methods ----------------

# Load saved summaries if needed
baseline_summary_path = RESULTS_DIR / f"baseline_summary_nclips{N_CLIPS}_top100.csv"
alpha_summary_path = RESULTS_DIR / f"alpha_fusion_summary_nclips{N_CLIPS}_top100.csv"
temporal_summary_path = RESULTS_DIR / f"temporal_smoothing_summary_nclips{N_CLIPS}_top100.csv"
candidate_summary_path = RESULTS_DIR / f"candidate_window_summary_nclips{N_CLIPS}_top100.csv"

print("Checking summary files:")
for p in [
    baseline_summary_path,
    alpha_summary_path,
    temporal_summary_path,
    candidate_summary_path,
]:
    print(p.name, "| exists:", p.exists())

baseline_summary = pd.read_csv(baseline_summary_path)
alpha_summary = pd.read_csv(alpha_summary_path)
temporal_summary = pd.read_csv(temporal_summary_path)
candidate_summary = pd.read_csv(candidate_summary_path)

metric_cols = [
    "top1_success",
    "top5_success",
    "top10_success",
    "top20_success",
    "top50_success",
    "top100_success",
]

# Baseline: keep CLIP and BLIP rows
baseline_rows = baseline_summary.copy()
baseline_rows["method"] = baseline_rows["model_name"] + "_baseline"
baseline_rows["setting"] = baseline_rows["model_name"]

baseline_rows = baseline_rows[
    ["method", "setting"] + metric_cols
]

# Alpha fusion: best by top100
alpha_best = (
    alpha_summary
    .sort_values(["top100_success", "top50_success", "top20_success", "top10_success"], ascending=False)
    .head(1)
    .copy()
)
alpha_best["method"] = "alpha_fusion"
alpha_best["setting"] = alpha_best.apply(
    lambda r: f"alpha={r['alpha']}",
    axis=1,
)
alpha_best = alpha_best[["method", "setting"] + metric_cols]

# Temporal smoothing: best by top100
temporal_best = (
    temporal_summary
    .sort_values(["top100_success", "top50_success", "top20_success", "top10_success"], ascending=False)
    .head(1)
    .copy()
)
temporal_best["method"] = "temporal_smoothing"
temporal_best["setting"] = temporal_best.apply(
    lambda r: (
        f"alpha={r['alpha']}, "
        f"window={int(r['window_size'])}, "
        f"mode={r['smoothing_mode']}"
    ),
    axis=1,
)
temporal_best = temporal_best[["method", "setting"] + metric_cols]

# Candidate-window: best by top100
candidate_best = (
    candidate_summary
    .sort_values(["top100_success", "top50_success", "top20_success", "top10_success"], ascending=False)
    .head(1)
    .copy()
)
candidate_best["method"] = "candidate_window"
candidate_best["setting"] = candidate_best.apply(
    lambda r: (
        f"alpha={r['alpha']}, "
        f"candidate_topk={int(r['candidate_topk'])}, "
        f"radius={int(r['window_radius'])}, "
        f"mode={r['score_mode']}"
    ),
    axis=1,
)
candidate_best = candidate_best[["method", "setting"] + metric_cols]

# Combine
final_comparison = pd.concat(
    [
        baseline_rows,
        alpha_best,
        temporal_best,
        candidate_best,
    ],
    ignore_index=True,
)

# Sort by top100
final_comparison = final_comparison.sort_values(
    ["top100_success", "top50_success", "top20_success", "top10_success"],
    ascending=False,
).reset_index(drop=True)

display(final_comparison)

final_comparison_path = RESULTS_DIR / f"final_method_comparison_nclips{N_CLIPS}_top100.csv"
final_comparison.to_csv(final_comparison_path, index=False)

print("Saved final comparison to:", final_comparison_path)

Checking summary files:
baseline_summary_nclips300_top100.csv | exists: True
alpha_fusion_summary_nclips300_top100.csv | exists: True
temporal_smoothing_summary_nclips300_top100.csv | exists: True
candidate_window_summary_nclips300_top100.csv | exists: True


,method,setting,top1_success,top5_success,top10_success,top20_success,top50_success,top100_success
0,candidate_window,"alpha=0.5, candidate_topk=50, radius=5, mode=max",0.605013,1.728608,2.765774,5.445117,13.828868,26.620570
1,alpha_fusion,alpha=0.5,0.605013,1.728608,2.765774,5.445117,13.828868,25.842697
2,temporal_smoothing,"alpha=0.5, window=7, mode=max",0.777874,1.987900,3.630078,7.000864,14.606742,25.756266
3,blip_baseline,blip,0.518583,1.469317,2.852204,5.617978,13.569576,25.669836
4,clip_baseline,clip,0.518583,1.728608,2.938634,5.272256,15.038894,24.978392


Saved final comparison to: /home/jupyter/results/retrieval_subset/final_method_comparison_nclips300_top100.csv


## Final Method Comparison

Among all tested retrieval strategies, candidate-guided window reranking achieved the best Top-100 performance. The best configuration used alpha = 0.5, candidate_topk = 50, window_radius = 5, and max aggregation. It reached 26.62% Top-100 accuracy, improving over the best alpha-fusion baseline of 25.84%.

Temporal smoothing did not achieve the best Top-100 performance, but it produced the strongest early-rank performance, reaching 3.63% Top-10 and 7.00% Top-20 accuracy. This suggests that temporal smoothing can help promote correct frames toward the top ranks, while candidate-guided window reranking is more effective for expanding the candidate set.

Overall, candidate-window reranking is selected as the main final method for Top-100 retrieval, while temporal smoothing provides a useful secondary insight for early-rank improvement.

In [ ]:
# ---------------- 36. Candidate-window + local temporal smoothing ----------------

def evaluate_one_clip_candidate_local_smoothing(
    clip_uid,
    alpha=0.5,
    candidate_topk=50,
    window_radius=5,
    local_smooth_radius=2,
    smooth_mode="max",
    final_topk=100,
    tolerance_frames=TOLERANCE_FRAMES,
):
    """
    Hybrid method:
    1. Compute CLIP/BLIP fusion scores.
    2. Select top candidate_topk anchor frames.
    3. Expand each anchor by +/- window_radius index positions.
    4. Only within expanded candidate frames, apply local temporal smoothing.
    5. Rank by locally smoothed scores.
    """

    index_cache = load_index_cache(clip_uid)
    query_cache = load_query_cache(clip_uid)

    frame_numbers = index_cache["frame_numbers"]

    index_clip_embs = index_cache["clip_embs"]
    index_blip_embs = index_cache["blip_embs"]

    query_clip_embs = query_cache["clip_embs"]
    query_blip_embs = query_cache["blip_embs"]

    qmeta = query_cache["query_metadata"].copy()
    qmeta["clip_uid"] = qmeta["clip_uid"].astype(str)
    qmeta["annotation_uid"] = qmeta["annotation_uid"].astype(str)
    qmeta["qs_id"] = qmeta["qs_id"].astype(int)

    rows = []

    for q_idx, qrow in qmeta.iterrows():
        key = (
            str(qrow["clip_uid"]),
            str(qrow["annotation_uid"]),
            int(qrow["qs_id"]),
        )

        if key not in rt_lookup:
            continue

        rt = rt_lookup[key]

        clip_scores = compute_similarity_scores(index_clip_embs, query_clip_embs[q_idx])
        blip_scores = compute_similarity_scores(index_blip_embs, query_blip_embs[q_idx])

        fusion_scores = alpha * clip_scores + (1 - alpha) * blip_scores

        # 1. Candidate anchors
        candidate_topk_actual = min(candidate_topk, len(fusion_scores))
        anchor_indices = np.argsort(-fusion_scores)[:candidate_topk_actual]

        # 2. Expanded candidate index set
        expanded_indices = set()

        for anchor_idx in anchor_indices:
            start = max(0, anchor_idx - window_radius)
            end = min(len(fusion_scores), anchor_idx + window_radius + 1)

            for idx in range(start, end):
                expanded_indices.add(idx)

        expanded_indices = sorted(expanded_indices)

        # 3. Local smoothing only for expanded indices
        rerank_rows = []

        for idx in expanded_indices:
            smooth_start = max(0, idx - local_smooth_radius)
            smooth_end = min(len(fusion_scores), idx + local_smooth_radius + 1)

            local_scores = fusion_scores[smooth_start:smooth_end]

            if smooth_mode == "max":
                smoothed_score = float(np.max(local_scores))
            elif smooth_mode == "mean":
                smoothed_score = float(np.mean(local_scores))
            else:
                raise ValueError("smooth_mode must be 'max' or 'mean'.")

            rerank_rows.append({
                "frame_number": int(frame_numbers[idx]),
                "score": smoothed_score,
                "index_position": int(idx),
            })

        topk_df = (
            pd.DataFrame(rerank_rows)
            .sort_values("score", ascending=False)
            .head(final_topk)
            .reset_index(drop=True)
        )

        topk_df["rank"] = np.arange(1, len(topk_df) + 1)

        rank = first_correct_rank(
            topk_df=topk_df,
            rt_min_frame=rt["rt_min_frame"],
            rt_max_frame=rt["rt_max_frame"],
            tolerance_frames=tolerance_frames,
        )

        rows.append({
            "clip_uid": clip_uid,
            "annotation_uid": qrow["annotation_uid"],
            "qs_id": int(qrow["qs_id"]),
            "object_title": qrow["object_title"],
            "method": "candidate_local_smoothing",
            "alpha": alpha,
            "candidate_topk": candidate_topk,
            "window_radius": window_radius,
            "local_smooth_radius": local_smooth_radius,
            "smooth_mode": smooth_mode,
            "num_expanded_frames": len(expanded_indices),
            "top1_frame": int(topk_df.iloc[0]["frame_number"]),
            "top1_score": float(topk_df.iloc[0]["score"]),
            "rt_min_frame": rt["rt_min_frame"],
            "rt_max_frame": rt["rt_max_frame"],
            "first_correct_rank": rank,
            "top1_success": bool(rank == 1),
            "top5_success": bool(pd.notna(rank) and rank <= 5),
            "top10_success": bool(pd.notna(rank) and rank <= 10),
            "top20_success": bool(pd.notna(rank) and rank <= 20),
            "top50_success": bool(pd.notna(rank) and rank <= 50),
            "top100_success": bool(pd.notna(rank) and rank <= 100),
        })

    return pd.DataFrame(rows)


print("Candidate-window + local smoothing function defined.")

Candidate-window + local smoothing function defined.


In [ ]:
# ---------------- 37. One-clip candidate-local-smoothing sanity check ----------------

test_clip_uid = all_300_clip_uids[0]

hybrid_test_df = evaluate_one_clip_candidate_local_smoothing(
    clip_uid=test_clip_uid,
    alpha=0.5,
    candidate_topk=50,
    window_radius=5,
    local_smooth_radius=2,
    smooth_mode="max",
    final_topk=100,
)

print("Test clip:", test_clip_uid)
print("Rows:", len(hybrid_test_df))
display(hybrid_test_df)

Test clip: dfe962ab-6aa7-4888-9378-796817a30ab6
Rows: 6


,clip_uid,annotation_uid,qs_id,object_title,method,alpha,candidate_topk,window_radius,local_smooth_radius,smooth_mode,...,top1_score,rt_min_frame,rt_max_frame,first_correct_rank,top1_success,top5_success,top10_success,top20_success,top50_success,top100_success
0,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,2,portable fan,candidate_local_smoothing,0.5,50,5,2,max,...,0.774597,42288.0,42317.0,42.0,False,False,False,False,True,True
1,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,3,tube,candidate_local_smoothing,0.5,50,5,2,max,...,0.708696,46764.0,46770.0,14.0,False,False,False,True,True,True
2,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,1,green bowl,candidate_local_smoothing,0.5,50,5,2,max,...,0.714960,49488.0,49536.0,NaN,False,False,False,False,False,False
3,dfe962ab-6aa7-4888-9378-796817a30ab6,dc29a4cf-1430-43fe-98d2-156ca2b303cf,2,stool,candidate_local_smoothing,0.5,50,5,2,max,...,0.777815,49494.0,49511.0,NaN,False,False,False,False,False,False
4,dfe962ab-6aa7-4888-9378-796817a30ab6,dc29a4cf-1430-43fe-98d2-156ca2b303cf,3,bucket,candidate_local_smoothing,0.5,50,5,2,max,...,0.786738,42678.0,42690.0,NaN,False,False,False,False,False,False
5,dfe962ab-6aa7-4888-9378-796817a30ab6,dc29a4cf-1430-43fe-98d2-156ca2b303cf,1,stool,candidate_local_smoothing,0.5,50,5,2,max,...,0.780019,43434.0,43441.0,9.0,False,False,True,True,True,True


In [ ]:
# ---------------- 38. Define hybrid candidate-local-smoothing grid ----------------

hybrid_grid = []

local_smooth_radii = [1, 2, 3, 5]
smooth_modes = ["max", "mean"]

for local_smooth_radius in local_smooth_radii:
    for smooth_mode in smooth_modes:
        hybrid_grid.append({
            "alpha": 0.5,
            "candidate_topk": 50,
            "window_radius": 5,
            "local_smooth_radius": local_smooth_radius,
            "smooth_mode": smooth_mode,
        })

hybrid_grid_df = pd.DataFrame(hybrid_grid)

print("Total hybrid settings:", len(hybrid_grid_df))
display(hybrid_grid_df)

Total hybrid settings: 8


,alpha,candidate_topk,window_radius,local_smooth_radius,smooth_mode
0,0.5,50,5,1,max
1,0.5,50,5,1,mean
2,0.5,50,5,2,max
3,0.5,50,5,2,mean
4,0.5,50,5,3,max
5,0.5,50,5,3,mean
6,0.5,50,5,5,max
7,0.5,50,5,5,mean


In [ ]:
# ---------------- 39. Run hybrid candidate-local-smoothing grid safely ----------------

HYBRID_RESULT_DIR = RESULTS_DIR / "candidate_local_smoothing_results"
HYBRID_RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("Hybrid result dir:", HYBRID_RESULT_DIR)

all_hybrid_result_paths = []

for cfg_idx, cfg in enumerate(hybrid_grid):
    alpha = cfg["alpha"]
    candidate_topk = cfg["candidate_topk"]
    window_radius = cfg["window_radius"]
    local_smooth_radius = cfg["local_smooth_radius"]
    smooth_mode = cfg["smooth_mode"]

    cfg_name = (
        f"candidate_local_smoothing_"
        f"alpha{alpha}_"
        f"topk{candidate_topk}_"
        f"radius{window_radius}_"
        f"localsmooth{local_smooth_radius}_"
        f"{smooth_mode}_"
        f"nclips{N_CLIPS}.csv"
    )

    cfg_path = HYBRID_RESULT_DIR / cfg_name
    all_hybrid_result_paths.append(cfg_path)

    print("=" * 100)
    print(f"[{cfg_idx + 1}/{len(hybrid_grid)}] Running:", cfg)
    print("Save path:", cfg_path)

    # Skip if already completed
    if cfg_path.exists():
        existing_df = pd.read_csv(cfg_path)
        if len(existing_df) == 1157:
            print("Already completed. Skipping.")
            continue
        else:
            print("Existing file incomplete. Recomputing:", len(existing_df))

    cfg_rows = []

    for clip_uid in tqdm(
        all_300_clip_uids,
        desc=f"local={local_smooth_radius}, mode={smooth_mode}"
    ):
        df_one = evaluate_one_clip_candidate_local_smoothing(
            clip_uid=clip_uid,
            alpha=alpha,
            candidate_topk=candidate_topk,
            window_radius=window_radius,
            local_smooth_radius=local_smooth_radius,
            smooth_mode=smooth_mode,
            final_topk=100,
        )
        cfg_rows.append(df_one)

    cfg_df = pd.concat(cfg_rows, ignore_index=True)

    print("Rows:", len(cfg_df))
    cfg_df.to_csv(cfg_path, index=False)
    print("Saved:", cfg_path)

print("All hybrid candidate-local-smoothing config runs finished or skipped.")

Hybrid result dir: /home/jupyter/results/retrieval_subset/candidate_local_smoothing_results
[1/8] Running: {'alpha': 0.5, 'candidate_topk': 50, 'window_radius': 5, 'local_smooth_radius': 1, 'smooth_mode': 'max'}
Save path: /home/jupyter/results/retrieval_subset/candidate_local_smoothing_results/candidate_local_smoothing_alpha0.5_topk50_radius5_localsmooth1_max_nclips300.csv


local=1, mode=max:   0%|          | 0/300 [00:00<?, ?it/s]

Rows: 1157
Saved: /home/jupyter/results/retrieval_subset/candidate_local_smoothing_results/candidate_local_smoothing_alpha0.5_topk50_radius5_localsmooth1_max_nclips300.csv
[2/8] Running: {'alpha': 0.5, 'candidate_topk': 50, 'window_radius': 5, 'local_smooth_radius': 1, 'smooth_mode': 'mean'}
Save path: /home/jupyter/results/retrieval_subset/candidate_local_smoothing_results/candidate_local_smoothing_alpha0.5_topk50_radius5_localsmooth1_mean_nclips300.csv


local=1, mode=mean:   0%|          | 0/300 [00:00<?, ?it/s]

Rows: 1157
Saved: /home/jupyter/results/retrieval_subset/candidate_local_smoothing_results/candidate_local_smoothing_alpha0.5_topk50_radius5_localsmooth1_mean_nclips300.csv
[3/8] Running: {'alpha': 0.5, 'candidate_topk': 50, 'window_radius': 5, 'local_smooth_radius': 2, 'smooth_mode': 'max'}
Save path: /home/jupyter/results/retrieval_subset/candidate_local_smoothing_results/candidate_local_smoothing_alpha0.5_topk50_radius5_localsmooth2_max_nclips300.csv


...

Rows: 1157
Saved: /home/jupyter/results/retrieval_subset/candidate_local_smoothing_results/candidate_local_smoothing_alpha0.5_topk50_radius5_localsmooth3_mean_nclips300.csv
[7/8] Running: {'alpha': 0.5, 'candidate_topk': 50, 'window_radius': 5, 'local_smooth_radius': 5, 'smooth_mode': 'max'}
Save path: /home/jupyter/results/retrieval_subset/candidate_local_smoothing_results/candidate_local_smoothing_alpha0.5_topk50_radius5_localsmooth5_max_nclips300.csv


local=5, mode=max:   0%|          | 0/300 [00:00<?, ?it/s]

Rows: 1157
Saved: /home/jupyter/results/retrieval_subset/candidate_local_smoothing_results/candidate_local_smoothing_alpha0.5_topk50_radius5_localsmooth5_max_nclips300.csv
[8/8] Running: {'alpha': 0.5, 'candidate_topk': 50, 'window_radius': 5, 'local_smooth_radius': 5, 'smooth_mode': 'mean'}
Save path: /home/jupyter/results/retrieval_subset/candidate_local_smoothing_results/candidate_local_smoothing_alpha0.5_topk50_radius5_localsmooth5_mean_nclips300.csv


local=5, mode=mean:   0%|          | 0/300 [00:00<?, ?it/s]

Rows: 1157
Saved: /home/jupyter/results/retrieval_subset/candidate_local_smoothing_results/candidate_local_smoothing_alpha0.5_topk50_radius5_localsmooth5_mean_nclips300.csv
All hybrid candidate-local-smoothing config runs finished or skipped.


In [ ]:
# ---------------- 40. Combine hybrid results and summarize ----------------

hybrid_result_files = sorted(HYBRID_RESULT_DIR.glob("candidate_local_smoothing_*.csv"))

print("Hybrid result files:", len(hybrid_result_files))
for p in hybrid_result_files:
    print(p.name)

hybrid_dfs = []

for p in hybrid_result_files:
    df = pd.read_csv(p)
    hybrid_dfs.append(df)

hybrid_df = pd.concat(hybrid_dfs, ignore_index=True)

print("Combined hybrid rows:", len(hybrid_df))
print("Expected rows:", 1157 * len(hybrid_result_files))

display(hybrid_df.head())

hybrid_all_path = RESULTS_DIR / f"candidate_local_smoothing_all_nclips{N_CLIPS}_top100.csv"
hybrid_df.to_csv(hybrid_all_path, index=False)

print("Saved combined hybrid results to:", hybrid_all_path)


metric_cols = [
    "top1_success",
    "top5_success",
    "top10_success",
    "top20_success",
    "top50_success",
    "top100_success",
]

hybrid_summary = (
    hybrid_df
    .groupby([
        "alpha",
        "candidate_topk",
        "window_radius",
        "local_smooth_radius",
        "smooth_mode",
    ])[metric_cols]
    .mean()
    .reset_index()
)

for col in metric_cols:
    hybrid_summary[col] = hybrid_summary[col] * 100

hybrid_summary_sorted = hybrid_summary.sort_values(
    ["top100_success", "top50_success", "top20_success", "top10_success"],
    ascending=False,
).reset_index(drop=True)

display(hybrid_summary_sorted)

hybrid_summary_path = RESULTS_DIR / f"candidate_local_smoothing_summary_nclips{N_CLIPS}_top100.csv"
hybrid_summary_sorted.to_csv(hybrid_summary_path, index=False)

print("Saved hybrid summary to:", hybrid_summary_path)

Hybrid result files: 8
candidate_local_smoothing_alpha0.5_topk50_radius5_localsmooth1_max_nclips300.csv
candidate_local_smoothing_alpha0.5_topk50_radius5_localsmooth1_mean_nclips300.csv
candidate_local_smoothing_alpha0.5_topk50_radius5_localsmooth2_max_nclips300.csv
candidate_local_smoothing_alpha0.5_topk50_radius5_localsmooth2_mean_nclips300.csv
candidate_local_smoothing_alpha0.5_topk50_radius5_localsmooth3_max_nclips300.csv
candidate_local_smoothing_alpha0.5_topk50_radius5_localsmooth3_mean_nclips300.csv
candidate_local_smoothing_alpha0.5_topk50_radius5_localsmooth5_max_nclips300.csv
candidate_local_smoothing_alpha0.5_topk50_radius5_localsmooth5_mean_nclips300.csv
Combined hybrid rows: 9256
Expected rows: 9256


,clip_uid,annotation_uid,qs_id,object_title,method,alpha,candidate_topk,window_radius,local_smooth_radius,smooth_mode,...,top1_score,rt_min_frame,rt_max_frame,first_correct_rank,top1_success,top5_success,top10_success,top20_success,top50_success,top100_success
0,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,2,portable fan,candidate_local_smoothing,0.5,50,5,1,max,...,0.774597,42288.0,42317.0,98.0,False,False,False,False,False,True
1,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,3,tube,candidate_local_smoothing,0.5,50,5,1,max,...,0.708696,46764.0,46770.0,9.0,False,False,True,True,True,True
2,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,1,green bowl,candidate_local_smoothing,0.5,50,5,1,max,...,0.714960,49488.0,49536.0,NaN,False,False,False,False,False,False
3,dfe962ab-6aa7-4888-9378-796817a30ab6,dc29a4cf-1430-43fe-98d2-156ca2b303cf,2,stool,candidate_local_smoothing,0.5,50,5,1,max,...,0.777815,49494.0,49511.0,NaN,False,False,False,False,False,False
4,dfe962ab-6aa7-4888-9378-796817a30ab6,dc29a4cf-1430-43fe-98d2-156ca2b303cf,3,bucket,candidate_local_smoothing,0.5,50,5,1,max,...,0.786738,42678.0,42690.0,79.0,False,False,False,False,False,True


Saved combined hybrid results to: /home/jupyter/results/retrieval_subset/candidate_local_smoothing_all_nclips300_top100.csv


,alpha,candidate_topk,window_radius,local_smooth_radius,smooth_mode,top1_success,top5_success,top10_success,top20_success,top50_success,top100_success
0,0.5,50,5,5,max,0.518583,2.074330,3.457217,7.692308,16.853933,27.917027
1,0.5,50,5,1,max,0.345722,2.074330,4.148660,5.877269,13.483146,25.756266
2,0.5,50,5,3,max,0.432152,1.901469,3.630078,7.260156,14.261020,25.669836
3,0.5,50,5,5,mean,0.086430,1.642178,2.852204,4.753673,12.878133,25.669836
4,0.5,50,5,2,max,0.777874,1.901469,3.975799,6.741573,13.310285,24.805532
5,0.5,50,5,1,mean,0.000000,1.210026,3.025065,5.012965,12.273120,24.546240
6,0.5,50,5,2,mean,0.259291,0.950735,2.247191,5.445117,11.840968,24.114088
7,0.5,50,5,3,mean,0.259291,1.210026,2.679343,5.099395,12.186690,23.681936


Saved hybrid summary to: /home/jupyter/results/retrieval_subset/candidate_local_smoothing_summary_nclips300_top100.csv


### Hybrid Candidate-Window with Local Temporal Smoothing

The best-performing method was the hybrid candidate-window plus local temporal smoothing strategy. The best configuration used alpha = 0.5, candidate_topk = 50, window_radius = 5, local_smooth_radius = 5, and max smoothing.

This method achieved 27.92% Top-100 accuracy, outperforming the best candidate-window-only result of 26.62% and the best alpha-fusion result of 25.84%. It also improved Top-20 accuracy to 7.69%, compared with 7.00% from temporal smoothing and 5.45% from candidate-window reranking.

These results suggest that candidate-window expansion improves candidate recall, while local temporal smoothing helps promote correct frames within candidate regions. The max smoothing strategy worked better than mean smoothing, indicating that preserving local peak scores is more effective than averaging nearby scores.

In [ ]:
# ---------------- 41. Final comparison including hybrid best ----------------

# Paths
baseline_summary_path = RESULTS_DIR / f"baseline_summary_nclips{N_CLIPS}_top100.csv"
alpha_summary_path = RESULTS_DIR / f"alpha_fusion_summary_nclips{N_CLIPS}_top100.csv"
temporal_summary_path = RESULTS_DIR / f"temporal_smoothing_summary_nclips{N_CLIPS}_top100.csv"
candidate_summary_path = RESULTS_DIR / f"candidate_window_summary_nclips{N_CLIPS}_top100.csv"
hybrid_summary_path = RESULTS_DIR / f"candidate_local_smoothing_summary_nclips{N_CLIPS}_top100.csv"

print("Checking summary files:")
for p in [
    baseline_summary_path,
    alpha_summary_path,
    temporal_summary_path,
    candidate_summary_path,
    hybrid_summary_path,
]:
    print(p.name, "| exists:", p.exists())

# Load summaries
baseline_summary = pd.read_csv(baseline_summary_path)
alpha_summary = pd.read_csv(alpha_summary_path)
temporal_summary = pd.read_csv(temporal_summary_path)
candidate_summary = pd.read_csv(candidate_summary_path)
hybrid_summary = pd.read_csv(hybrid_summary_path)

metric_cols = [
    "top1_success",
    "top5_success",
    "top10_success",
    "top20_success",
    "top50_success",
    "top100_success",
]

# 1. Baseline rows
baseline_rows = baseline_summary.copy()
baseline_rows["method"] = baseline_rows["model_name"] + "_baseline"
baseline_rows["setting"] = baseline_rows["model_name"]
baseline_rows = baseline_rows[["method", "setting"] + metric_cols]

# 2. Alpha fusion best
alpha_best = (
    alpha_summary
    .sort_values(
        ["top100_success", "top50_success", "top20_success", "top10_success"],
        ascending=False,
    )
    .head(1)
    .copy()
)
alpha_best["method"] = "alpha_fusion"
alpha_best["setting"] = alpha_best.apply(
    lambda r: f"alpha={r['alpha']}",
    axis=1,
)
alpha_best = alpha_best[["method", "setting"] + metric_cols]

# 3. Temporal smoothing best
temporal_best = (
    temporal_summary
    .sort_values(
        ["top100_success", "top50_success", "top20_success", "top10_success"],
        ascending=False,
    )
    .head(1)
    .copy()
)
temporal_best["method"] = "temporal_smoothing"
temporal_best["setting"] = temporal_best.apply(
    lambda r: (
        f"alpha={r['alpha']}, "
        f"window={int(r['window_size'])}, "
        f"mode={r['smoothing_mode']}"
    ),
    axis=1,
)
temporal_best = temporal_best[["method", "setting"] + metric_cols]

# 4. Candidate-window best
candidate_best = (
    candidate_summary
    .sort_values(
        ["top100_success", "top50_success", "top20_success", "top10_success"],
        ascending=False,
    )
    .head(1)
    .copy()
)
candidate_best["method"] = "candidate_window"
candidate_best["setting"] = candidate_best.apply(
    lambda r: (
        f"alpha={r['alpha']}, "
        f"candidate_topk={int(r['candidate_topk'])}, "
        f"radius={int(r['window_radius'])}, "
        f"mode={r['score_mode']}"
    ),
    axis=1,
)
candidate_best = candidate_best[["method", "setting"] + metric_cols]

# 5. Hybrid best
hybrid_best = (
    hybrid_summary
    .sort_values(
        ["top100_success", "top50_success", "top20_success", "top10_success"],
        ascending=False,
    )
    .head(1)
    .copy()
)
hybrid_best["method"] = "candidate_local_smoothing"
hybrid_best["setting"] = hybrid_best.apply(
    lambda r: (
        f"alpha={r['alpha']}, "
        f"candidate_topk={int(r['candidate_topk'])}, "
        f"window_radius={int(r['window_radius'])}, "
        f"local_smooth_radius={int(r['local_smooth_radius'])}, "
        f"mode={r['smooth_mode']}"
    ),
    axis=1,
)
hybrid_best = hybrid_best[["method", "setting"] + metric_cols]

# Combine all methods
final_comparison_with_hybrid = pd.concat(
    [
        baseline_rows,
        alpha_best,
        temporal_best,
        candidate_best,
        hybrid_best,
    ],
    ignore_index=True,
)

final_comparison_with_hybrid = final_comparison_with_hybrid.sort_values(
    ["top100_success", "top50_success", "top20_success", "top10_success"],
    ascending=False,
).reset_index(drop=True)

display(final_comparison_with_hybrid)

# Save
final_comparison_with_hybrid_path = (
    RESULTS_DIR / f"final_method_comparison_with_hybrid_nclips{N_CLIPS}_top100.csv"
)

final_comparison_with_hybrid.to_csv(
    final_comparison_with_hybrid_path,
    index=False,
)

print("Saved final comparison with hybrid to:", final_comparison_with_hybrid_path)

Checking summary files:
baseline_summary_nclips300_top100.csv | exists: True
alpha_fusion_summary_nclips300_top100.csv | exists: True
temporal_smoothing_summary_nclips300_top100.csv | exists: True
candidate_window_summary_nclips300_top100.csv | exists: True
candidate_local_smoothing_summary_nclips300_top100.csv | exists: True


,method,setting,top1_success,top5_success,top10_success,top20_success,top50_success,top100_success
0,candidate_local_smoothing,"alpha=0.5, candidate_topk=50, window_radius=5,...",0.518583,2.074330,3.457217,7.692308,16.853933,27.917027
1,candidate_window,"alpha=0.5, candidate_topk=50, radius=5, mode=max",0.605013,1.728608,2.765774,5.445117,13.828868,26.620570
2,alpha_fusion,alpha=0.5,0.605013,1.728608,2.765774,5.445117,13.828868,25.842697
3,temporal_smoothing,"alpha=0.5, window=7, mode=max",0.777874,1.987900,3.630078,7.000864,14.606742,25.756266
4,blip_baseline,blip,0.518583,1.469317,2.852204,5.617978,13.569576,25.669836
5,clip_baseline,clip,0.518583,1.728608,2.938634,5.272256,15.038894,24.978392


Saved final comparison with hybrid to: /home/jupyter/results/retrieval_subset/final_method_comparison_with_hybrid_nclips300_top100.csv


# Final Experiment Summary

We evaluated several retrieval strategies on a 300-clip subset using cached CLIP and BLIP embeddings.

The experiment followed a progressive retrieval pipeline:

1. Single-model baseline retrieval using CLIP and BLIP.
2. Score-level CLIP/BLIP alpha fusion.
3. Temporal smoothing as a post-processing strategy.
4. Candidate-guided window reranking.
5. Hybrid candidate-window reranking with local temporal smoothing.

The best final method was **candidate-guided local temporal smoothing**, using:

- alpha = 0.5
- candidate_topk = 50
- window_radius = 5
- local_smooth_radius = 5
- smooth_mode = max

This method achieved the best overall performance:

- Top20 accuracy: approximately 7.69%
- Top50 accuracy: approximately 16.85%
- Top100 accuracy: approximately 27.92%

The results show that candidate-window expansion improves candidate recall, while local temporal smoothing improves ranking within the expanded candidate regions. The hybrid method therefore provides the best balance between broad candidate retrieval and early-rank accuracy.